# 02 - Data Preprocessing
**Project:** Predictive Modelling of Residential Property Prices (Ames Housing) | IT3091, group 2026-DS-38
**Input:** `Dataset/train.csv` (1,460 labelled sales). **Output:** a reusable, leakage-safe preprocessing pipeline, `build_preprocessor()` in `src/features/preprocessing.py`.

## 1. Introduction
**Purpose.** Raw property records cannot be given directly to a regression model: they mix numeric and categorical variables, contain gaps whose meaning differs by column, and include variables that would not be known when a listing price is set. Preprocessing converts them into a consistent numeric matrix without changing what the data says.

**Why it matters for house prices.** Price is strongly driven by quality, size and neighbourhood, so errors in how these are represented (a missing basement treated as an average one, an unordered quality rating treated as a number, a neighbourhood given an arbitrary rank) feed straight into the estimate and into the prediction interval the project depends on.

**Missing values need different treatments.** The exploratory analysis (`notebooks/01_eda.ipynb`, `docs/eda_insight_log.md`) showed that 15 of the 19 columns containing "NA" describe a facility that does not exist (no garage, no pool), not an unknown value. Only `LotFrontage`, `MasVnrType`/`MasVnrArea` and `Electrical` are genuinely missing. A single imputation rule would therefore corrupt some columns and be wasteful for others.

**Avoiding leakage.** Any statistic learned from the data (a median, a mode, nearest-neighbour donors, category lists, scaling parameters) must be learned from the training rows of each cross-validation fold only. Otherwise information from validation rows leaks into training and the reported accuracy is inflated. All steps are therefore built as scikit-learn pipeline components and are never fitted on the full dataset before evaluation.

**How this supports modelling.** The notebook ends with `build_preprocessor(strategy)`, which any model can be attached to. The EDA is not repeated here; its findings are cited as evidence for each decision.

In [1]:
import sys, inspect, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
ROOT = Path.cwd().resolve()
while not (ROOT / 'CLAUDE.md').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from src.data.loading import load_raw, TEST_PATH
from src.features import preprocessing as P
from src.features.preprocessing import (build_preprocessor, StructuralAbsenceImputer, NeighborhoodMedianImputer,
                                        GlobalMedianImputer, KNNFrontageImputer, AddMissingIndicator, PropertyFeatureBuilder)
from src.evaluation.cv import get_folds, cv_scores, summarize, paired_difference

SEED = 42
np.random.seed(SEED)
TAB = ROOT / 'reports' / 'tables'; TAB.mkdir(parents=True, exist_ok=True)
pd.set_option('display.width', 200, 'display.max_columns', 40, 'display.max_rows', 80, 'display.max_colwidth', 70)

## 2. Load Data and Verify Structure
**Issue.** Before transforming anything we need the starting point: the dimensions, variable types, amount of missingness, duplicates and the target.
**Method.** The data is loaded with `load_raw()`, which keeps the string `NA` only in the 14 columns where the data description defines it as "feature absent" and turns it into a real missing value elsewhere (the EDA found that plain `pd.read_csv` cannot tell the two apart). The original frame `df` is never modified; `X` and `y` are separate copies.

In [2]:
df = load_raw()                                   # original data - not modified below
test_df = load_raw(TEST_PATH)                     # used only to check that the pipeline can transform unseen records
X = df.drop(columns=['SalePrice', 'Id'])
y = df['SalePrice']
assert 'SalePrice' not in X.columns and 'Id' not in X.columns, 'target or identifier leaked into predictors'

na_marker = int(sum((X[c] == 'NA').sum() for c in P.STRUCTURAL_COLUMNS))
summary = pd.Series({
    'Rows': len(df), 'Columns in file (incl. Id, SalePrice)': df.shape[1], 'Predictors in X': X.shape[1],
    'Numeric-dtype predictors (incl. MSSubClass code)': X.select_dtypes('number').shape[1],
    'Categorical-dtype predictors': X.select_dtypes('object').shape[1],
    'Genuinely missing cells (NaN) in X': int(X.isna().sum().sum()),
    "'NA' cells meaning feature absent (14 columns)": na_marker,
    'Duplicate rows (excluding Id)': int(X.duplicated().sum()),
    'Target missing values': int(y.isna().sum()),
    'Target min / median / max ($)': f'{y.min():,.0f} / {y.median():,.0f} / {y.max():,.0f}',
    'Target skew raw -> log1p': f'{y.skew():.2f} -> {np.log1p(y).skew():.2f}',
}, name='value').to_frame()
summary.to_csv(TAB / 'prep_start_summary.csv')
summary

,value
Rows,1460
"Columns in file (incl. Id, SalePrice)",81
Predictors in X,79
Numeric-dtype predictors (incl. MSSubClass code),36
Categorical-dtype predictors,43
Genuinely missing cells (NaN) in X,357
'NA' cells meaning feature absent (14 columns),6608
Duplicate rows (excluding Id),0
Target missing values,0
Target min / median / max ($),"34,900 / 163,000 / 755,000"


**Interpretation.** The file holds 1,460 sales and 79 predictors (36 numeric-dtype, 43 categorical; the numeric `MSSubClass` is really a category). The target has no missing values and the data has no duplicate rows. The count of 357 genuinely missing cells is much smaller than the 6,965 cells that plain pandas would report as missing: a further 6,608 `NA` cells in 14 columns mean "facility absent" and are not gaps. `SalePrice` is strongly skewed (1.88, reduced to 0.12 by `log1p`), so all models are fitted to `log1p(SalePrice)`. The 15 folds are created once and reused by every experiment below.

In [3]:
# One set of folds, reused by every experiment below: repeated 5-fold CV (3 repeats), stratified by price band.
folds = get_folds(y, n_splits=5, n_repeats=3, seed=SEED)
tr0, va0 = folds[0]                       # fold 1 is used wherever a fitted object is displayed
print(len(folds), 'folds | fold-1 train/validation rows:', len(tr0), len(va0))

15 folds | fold-1 train/validation rows: 1168 292


## 3. Missing-Value Analysis
**Issue.** Missing entries carry different meanings, so each column must be classified before it is treated.
**Method.** Each column is assigned to one of three classes, using `data_description.txt` and the related numeric columns:
- **A. Structural:** missing means the facility does not exist (confirmed by a related column such as `GarageArea = 0`).
- **B. Genuine:** the feature exists, or must exist, but its value is unknown.
- **C. Ambiguous / inconsistent:** related columns conflict, or do not establish whether the facility exists.

In [4]:
absence = StructuralAbsenceImputer().fit(X).classify(X)       # per structural column: confirmed absent vs facility exists
meaning = {
 'PoolQC': ('No pool', 'NoFeature (confirmed by PoolArea = 0)', 'Explicit category; the most frequent value would give 99.5% of houses a pool'),
 'MiscFeature': ('No miscellaneous feature', 'NoFeature (confirmed by MiscVal = 0)', 'Absence is information, not noise'),
 'Alley': ('No alley access', 'NoFeature (documented; no related column)', 'data_description.txt defines NA as no alley access'),
 'Fence': ('No fence', 'NoFeature (documented; no related column)', 'data_description.txt defines NA as no fence'),
 'FireplaceQu': ('No fireplace', 'NoFeature (confirmed by Fireplaces = 0)', 'Matches Fireplaces = 0 on all 690 rows'),
 'GarageType': ('No garage', 'NoFeature (GarageArea = GarageCars = 0)', 'Matches zero garage size on all 81 rows'),
 'GarageFinish': ('No garage', 'NoFeature (GarageArea = GarageCars = 0)', 'As above'),
 'GarageQual': ('No garage', 'NoFeature (GarageArea = GarageCars = 0)', 'As above'),
 'GarageCond': ('No garage', 'NoFeature (GarageArea = GarageCars = 0)', 'As above'),
 'BsmtQual': ('No basement', 'NoFeature (TotalBsmtSF = 0)', 'Matches TotalBsmtSF = 0 on all 37 rows'),
 'BsmtCond': ('No basement', 'NoFeature (TotalBsmtSF = 0)', 'As above'),
 'BsmtExposure': ('No basement (37 rows); 1 row has a basement', 'NoFeature if TotalBsmtSF = 0, else most frequent among real basements', 'Id 949 has a 936 sq ft basement but no exposure recorded'),
 'BsmtFinType1': ('No basement', 'NoFeature (TotalBsmtSF = 0)', 'As above'),
 'BsmtFinType2': ('No basement (37 rows); 1 row has a basement', 'NoFeature if TotalBsmtSF = 0, else most frequent among real basements', 'Id 333 has a 3,206 sq ft basement but no second finish type'),
}
rows = []
for c in P.STRUCTURAL_COLUMNS:
    n, conf, unk = absence.loc[c]
    cls = 'A' if unk == 0 else 'A + C'
    m, t, r = meaning[c]
    rows.append((c, n, round(n / len(X) * 100, 1), 'object', m, cls, t, r))
rows += [
 ('GarageYrBlt', int(X.GarageYrBlt.isna().sum()), round(X.GarageYrBlt.isna().mean() * 100, 1), 'float64', 'No garage (NaN on exactly the 81 no-garage houses)', 'A',
  'Not used as a predictor', 'Redundant with YearBuilt (r = 0.83) and 9 rows are dated before the house (EDA F16, F19)'),
 ('LotFrontage', int(X.LotFrontage.isna().sum()), round(X.LotFrontage.isna().mean() * 100, 1), 'float64', 'Frontage exists but was not recorded', 'B',
  'Median / neighbourhood median / KNN (compared)', 'Not missing at random: depends on lot shape, configuration and neighbourhood (EDA F07)'),
 ('MasVnrType', int(X.MasVnrType.isna().sum()), round(X.MasVnrType.isna().mean() * 100, 1), 'object', 'Veneer type unrecorded ("None" is a real category, 864 rows)', 'B',
  'Most frequent category', '8 rows only; the same rows as MasVnrArea'),
 ('MasVnrArea', int(X.MasVnrArea.isna().sum()), round(X.MasVnrArea.isna().mean() * 100, 1), 'float64', 'Veneer area unrecorded', 'B',
  'Median', 'Median is 0 (59% of houses have no veneer); the mean would invent a veneer'),
 ('Electrical', int(X.Electrical.isna().sum()), round(X.Electrical.isna().mean() * 100, 1), 'object', 'Electrical system unrecorded (Id 1380)', 'B',
  'Most frequent category', '1 row; 91% of houses have SBrkr'),
]
miss = pd.DataFrame(rows, columns=['Feature', 'Missing Count', 'Missing Percentage', 'Data Type', 'Missingness Meaning', 'Class', 'Candidate Treatment', 'Reason'])
miss = miss.sort_values('Missing Count', ascending=False).reset_index(drop=True)
miss.to_csv(TAB / 'prep_missing_summary.csv', index=False)
miss

,Feature,Missing Count,Missing Percentage,Data Type,Missingness Meaning,Class,Candidate Treatment,Reason
0,PoolQC,1453,99.5,object,No pool,A,NoFeature (confirmed by PoolArea = 0),Explicit category; the most frequent value would give 99.5% of hou...
1,MiscFeature,1406,96.3,object,No miscellaneous feature,A,NoFeature (confirmed by MiscVal = 0),"Absence is information, not noise"
2,Alley,1369,93.8,object,No alley access,A,NoFeature (documented; no related column),data_description.txt defines NA as no alley access
3,Fence,1179,80.8,object,No fence,A,NoFeature (documented; no related column),data_description.txt defines NA as no fence
4,FireplaceQu,690,47.3,object,No fireplace,A,NoFeature (confirmed by Fireplaces = 0),Matches Fireplaces = 0 on all 690 rows
5,LotFrontage,259,17.7,float64,Frontage exists but was not recorded,B,Median / neighbourhood median / KNN (compared),"Not missing at random: depends on lot shape, configuration and nei..."
6,GarageFinish,81,5.5,object,No garage,A,NoFeature (GarageArea = GarageCars = 0),As above
7,GarageYrBlt,81,5.5,float64,No garage (NaN on exactly the 81 no-garage houses),A,Not used as a predictor,Redundant with YearBuilt (r = 0.83) and 9 rows are dated before th...
8,GarageType,81,5.5,object,No garage,A,NoFeature (GarageArea = GarageCars = 0),Matches zero garage size on all 81 rows
9,GarageCond,81,5.5,object,No garage,A,NoFeature (GarageArea = GarageCars = 0),As above


**Interpretation.** Nineteen columns are affected. Fifteen are structural (class A): the 14 `NA` category columns plus `GarageYrBlt`, whose missing values coincide exactly with the 81 houses without a garage. Of these, `BsmtExposure` and `BsmtFinType2` each also contain one inconsistent record (class C: Ids 949 and 333 have a basement but one rating is `NA`). Only four columns are genuinely missing (class B): `LotFrontage` (17.7%), `MasVnrType`/`MasVnrArea` (the same 8 rows) and `Electrical` (1 row). Because the classes differ in meaning, no single imputation rule is appropriate; the treatments in the following sections are matched to each class.

## 4. Structural Missing Values
**Issue.** In 14 columns `NA` can mean "the property has no such facility". Replacing it with the most frequent value would give a house with no pool a pool.
**Method.** Absence is only accepted when it is *verified*: the related numeric columns are all zero (garage: `GarageArea` and `GarageCars`; basement: `TotalBsmtSF`; fireplace: `Fireplaces`; pool: `PoolArea`; misc: `MiscVal`), or, where the numeric columns are themselves missing, every category in the group is `NA`. `Alley` and `Fence` have no related column, so the documented meaning is used. Confirmed absence becomes the explicit category `NoFeature`, and the matching numeric columns become 0 (Case A). If `NA` appears on a property that does have the facility, the value is genuinely unknown and is filled with the most frequent category *among properties that really have the facility*, never with `NoFeature`.

In [5]:
def evidence_table(d):
    rows = []
    for g, spec in P.STRUCTURAL_GROUPS.items():
        if spec['evidence'] is None:
            continue
        allna = pd.concat([(d[c] == 'NA') for c in spec['cats']], axis=1).all(axis=1)
        anyna = pd.concat([(d[c] == 'NA') for c in spec['cats']], axis=1).any(axis=1)
        zero = (d[spec['evidence']] == 0).all(axis=1)
        rows.append((g, int(allna.sum()), int((anyna & zero).sum()), int((anyna & ~zero).sum()), int((~anyna & zero).sum())))
    return pd.DataFrame(rows, columns=['group', 'rows with all categories NA', 'NA & numeric evidence = 0 (absent)',
                                       'NA but facility exists (contradiction)', 'value present but evidence = 0']).set_index('group')
print('TRAIN'); display(evidence_table(X))
print('TEST (used only to check that unusual records are handled)'); display(evidence_table(test_df))

TRAIN


,rows with all categories NA,NA & numeric evidence = 0 (absent),NA but facility exists (contradiction),value present but evidence = 0
group,,,,
garage,81,81,0,0
basement,37,37,2,0
fireplace,690,690,0,0
pool,1453,1453,0,0
misc,1406,1406,0,2


TEST (used only to check that unusual records are handled)


,rows with all categories NA,NA & numeric evidence = 0 (absent),NA but facility exists (contradiction),value present but evidence = 0
group,,,,
garage,76,76,2,0
basement,42,41,8,0
fireplace,730,730,0,0
pool,1456,1453,3,0
misc,1408,1407,1,1


In [6]:
# Contradictory records
show = ['BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'TotalBsmtSF']
print('Train: basement exists but a category is NA'); display(df.loc[df.Id.isin([333, 949]), ['Id'] + show])
print('Test examples: garage exists but categories NA / area unknown; pool exists but quality NA; no basement info at all')
chk = test_df[test_df.Id.isin([2127, 2577, 2421, 2121])]
display(chk[['Id', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageArea', 'GarageCars', 'PoolQC', 'PoolArea', 'BsmtQual', 'TotalBsmtSF']])
resolved = StructuralAbsenceImputer().fit(X).transform(test_df[test_df.Id.isin([2127, 2577, 2421, 2121])])
print('After StructuralAbsenceImputer (fitted on training data):')
display(resolved[['Id', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageArea', 'GarageCars', 'PoolQC', 'PoolArea', 'BsmtQual', 'TotalBsmtSF']])

Train: basement exists but a category is NA


,Id,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinType2,TotalBsmtSF
332,333,Gd,TA,No,GLQ,NA,3206
948,949,Gd,TA,NA,Unf,Unf,936


Test examples: garage exists but categories NA / area unknown; pool exists but quality NA; no basement info at all


,Id,GarageType,GarageFinish,GarageQual,GarageArea,GarageCars,PoolQC,PoolArea,BsmtQual,TotalBsmtSF
660,2121,Detchd,Unf,TA,280.0,1.0,NA,0,NA,NaN
666,2127,Detchd,NA,NA,360.0,1.0,NA,0,TA,1242.0
960,2421,Attchd,Fin,TA,280.0,1.0,NA,368,TA,1105.0
1116,2577,Detchd,NA,NA,NaN,NaN,NA,0,Gd,859.0


After StructuralAbsenceImputer (fitted on training data):


,Id,GarageType,GarageFinish,GarageQual,GarageArea,GarageCars,PoolQC,PoolArea,BsmtQual,TotalBsmtSF
660,2121,Detchd,Unf,TA,280.0,1.0,NoFeature,0,NoFeature,0.0
666,2127,Detchd,Unf,TA,360.0,1.0,NoFeature,0,TA,1242.0
960,2421,Attchd,Fin,TA,280.0,1.0,Gd,368,TA,1105.0
1116,2577,Detchd,Unf,TA,NaN,NaN,NoFeature,0,Gd,859.0


In [7]:
# Why most-frequent imputation would be wrong for structural absence
naive_pool = X.PoolQC.replace('NA', np.nan).mode()[0]
print(f'Naive mode of PoolQC among recorded values: {naive_pool!r} -> would assign a pool to {(X.PoolQC == "NA").mean():.1%} of houses')
naive_gar = X.GarageType.replace('NA', np.nan).mode()[0]
print(f'Naive mode of GarageType: {naive_gar!r} -> would give a garage to the {int((X.GarageType == "NA").sum())} houses with GarageArea = 0')
print(); print(inspect.getsource(StructuralAbsenceImputer))

Naive mode of PoolQC among recorded values: 'Gd' -> would assign a pool to 99.5% of houses
Naive mode of GarageType: 'Attchd' -> would give a garage to the 81 houses with GarageArea = 0

class StructuralAbsenceImputer(BaseEstimator, TransformerMixin):
    """Resolve the string 'NA' in the 14 structural columns.

    * Absence is *confirmed* when related numeric columns are all 0 (e.g. GarageArea = GarageCars = 0),
      or when the numeric columns are themselves missing and every category in the group is 'NA',
      or (Alley, Fence) by documentation. Confirmed absence -> 'NoFeature' and missing related numerics -> 0.
    * 'NA' on a property that has the facility is *genuinely unknown*. It is replaced by the most frequent
      category among training rows that really have the facility (never by 'NoFeature').
    * MasVnrArea is 0 when MasVnrType is 'None'.
    """

    def __init__(self, groups=None):
        self.groups = groups

    def _groups(self):
        return self.groups or 

**Interpretation.** In the training data every `NA` in the garage, basement, fireplace and pool groups coincides with a zero in the related numeric column (81, 37, 690 and 1,453 rows), so these are confirmed absences. Contradictions exist: two training rows (Ids 333 and 949) have a basement with one rating missing, and the test file contains 14 group-level cases (2 garage, 8 basement, 3 pool, 1 misc) that the transformer must handle. They are treated as *unknown values of an existing facility*: Id 2127 receives the typical finish and quality of an existing detached garage (`Unf`, `TA`), Id 2421 receives the typical quality among houses that really have a pool (`Gd`), and neither is labelled `NoFeature`. Id 2121 has no basement information at all and is treated as having none (`NoFeature`, areas 0). Plain mode imputation would have given 99.5% of houses a pool and 81 houses without a garage a garage type, which is why structural absence is never filled with the most frequent category. Two training rows record a `MiscFeature` with `MiscVal = 0`; they are left unchanged. `Alley` and `Fence` cannot be cross-checked against another column, so their absence rests on `data_description.txt`.

## 5. Numerical Missing Values
**Issue.** A missing number can mean either that the feature does not exist (the value should be 0) or that it exists but was not measured (the value must be estimated). Zero-filling the second case would invent a house with no garage.
**Method.** *Case A* (verified absence) is handled by `StructuralAbsenceImputer`, which sets the related numeric columns to 0. *Case B* (feature exists, value unknown) is left missing for a **median** imputer (`SimpleImputer(strategy="median")`) inside the pipeline, so the median is learned from the training fold only. The median is preferred to the mean because property measurements are skewed or contain extreme values: a few very large lots or veneers pull the mean away from a typical property.

In [8]:
num_miss = pd.DataFrame({'train NaN': X.select_dtypes('number').isna().sum(), 'test NaN': test_df.drop(columns='SalePrice', errors='ignore').select_dtypes('number').isna().sum()})
num_miss = num_miss[(num_miss > 0).any(axis=1)]
display(num_miss)
print('GarageArea / GarageCars missing in train:', int(X.GarageArea.isna().sum()), '/', int(X.GarageCars.isna().sum()),
      '-> Case B cannot occur in train; it occurs in test (Id 2577: detached garage, area and cars unknown)')
t = test_df[test_df.Id.isin([2577, 2121])]
r = StructuralAbsenceImputer().fit(X).transform(t)
print('Case B (Id 2577: garage exists, area stays NaN for the median imputer) and Case A (Id 2121: no basement, areas set to 0):')
display(r[['Id', 'GarageType', 'GarageArea', 'GarageCars', 'BsmtQual', 'TotalBsmtSF', 'BsmtFinSF1', 'BsmtFullBath']])

,train NaN,test NaN
BsmtFinSF1,0.0,1
BsmtFinSF2,0.0,1
BsmtFullBath,0.0,2
BsmtHalfBath,0.0,2
BsmtUnfSF,0.0,1
GarageArea,0.0,1
GarageCars,0.0,1
GarageYrBlt,81.0,78
LotFrontage,259.0,227
MasVnrArea,8.0,15


GarageArea / GarageCars missing in train: 0 / 0 -> Case B cannot occur in train; it occurs in test (Id 2577: detached garage, area and cars unknown)
Case B (Id 2577: garage exists, area stays NaN for the median imputer) and Case A (Id 2121: no basement, areas set to 0):


,Id,GarageType,GarageArea,GarageCars,BsmtQual,TotalBsmtSF,BsmtFinSF1,BsmtFullBath
660,2121,Detchd,280.0,1.0,NoFeature,0.0,0.0,0.0
1116,2577,Detchd,NaN,NaN,Gd,859.0,548.0,0.0


In [9]:
# Why median rather than mean (training data)
cols = ['LotFrontage', 'MasVnrArea', 'LotArea']
stat = pd.DataFrame({'mean': X[cols].mean(), 'median': X[cols].median(), 'max': X[cols].max(), 'skew': X[cols].skew(), '% zeros': (X[cols] == 0).mean() * 100}).round(2)
display(stat)
imp = SimpleImputer(strategy='median').fit(X.iloc[tr0][['LotFrontage', 'MasVnrArea']])
print('Medians learned from fold-1 training rows only:', dict(zip(['LotFrontage', 'MasVnrArea'], imp.statistics_)))

,mean,median,max,skew,% zeros
LotFrontage,70.05,69.0,313.0,2.16,0.00
MasVnrArea,103.69,0.0,1600.0,2.67,58.97
LotArea,10516.83,9478.5,215245.0,12.21,0.00


Medians learned from fold-1 training rows only: {'LotFrontage': 70.0, 'MasVnrArea': 0.0}


**Interpretation.** Training data contains genuinely missing numbers only in `LotFrontage` (259), `MasVnrArea` (8) and `GarageYrBlt` (81, structural; the column is not used as a predictor). `GarageArea` and `GarageCars` have no missing values in the training data, so Case B for garage size does not arise there; it does in the test file (Id 2577: a detached garage of unknown size), where the value is deliberately left missing for the median imputer instead of being set to 0, while Id 2121 (no basement) correctly receives zeros (Case A). The median is the right baseline: the mean veneer area (103.7 sq ft) would give a house with no recorded veneer a veneer, because 59% of houses have none and the median is 0; `LotArea` is extremely skewed (12.2; mean 10,517 vs median 9,478). For `LotFrontage` the mean (70.1) and median (69) are close, so the median is chosen for consistency and robustness. The medians are learned from the training fold (70 and 0 in fold 1).

## 6. Categorical Mode Imputation
**Issue.** A few categorical variables have a handful of genuinely missing values once structural absence has been resolved.
**Method.** `SimpleImputer(strategy="most_frequent")` fitted on the training fold. For a categorical variable the mode is the only central value that is a valid category (a mean or median is undefined), and when very few values are missing a replacement by the most common category is the least disruptive choice. It is an assumption, not a recovery of the true value. It is **not** used for structural absence (section 4).

In [10]:
cat_train = X.select_dtypes('object')
tst = test_df.drop(columns='SalePrice', errors='ignore')
resolved_train = StructuralAbsenceImputer().fit(X).transform(X)
resolved_test = StructuralAbsenceImputer().fit(X).transform(tst)
rows = []
for c in sorted(set(resolved_train.columns)):
    if resolved_train[c].dtype != object and not (c in P.ORDINAL_ORDERS or c in P.NOMINAL):
        continue
    ntr, nte = int(resolved_train[c].isna().sum()), int(resolved_test[c].isna().sum())
    if ntr or nte:
        vc = resolved_train[c].value_counts(normalize=True)
        rows.append((c, ntr, nte, vc.index[0], round(vc.iloc[0] * 100, 1)))
mode_tab = pd.DataFrame(rows, columns=['feature', 'missing in train', 'missing in test', 'training mode', 'mode share %']).set_index('feature')
mode_tab.loc[[i for i in mode_tab.index if i not in P.SALE_CONTEXT]]

,missing in train,missing in test,training mode,mode share %
feature,,,,
Electrical,1,0,SBrkr,91.4
Exterior1st,0,1,VinylSd,35.3
Exterior2nd,0,1,VinylSd,34.5
Functional,0,2,Typ,93.2
KitchenQual,0,1,TA,50.3
MSZoning,0,4,RL,78.8
MasVnrType,8,16,None,59.5
Utilities,0,2,AllPub,99.9


In [11]:
# Mode vs an explicit "Unknown" level for the one training row with missing Electrical (Id 1380)
row = df[df.Id == 1380][['Id', 'Electrical', 'YearBuilt', 'Neighborhood']]; display(row)
same_era = df[(df.YearBuilt >= 2000)].Electrical.value_counts(normalize=True).round(3)
print('Electrical among houses built 2000 or later:', same_era.to_dict())
imp = SimpleImputer(strategy='most_frequent').fit(X[['Electrical', 'MSZoning', 'KitchenQual']].iloc[tr0])
print('Modes learned from fold-1 training rows only:', dict(zip(['Electrical', 'MSZoning', 'KitchenQual'], imp.statistics_)))
print('An "Unknown" level would be a column with', int(X.Electrical.isna().sum()), 'training row(s); OneHotEncoder(min_frequency=10) would merge it into "infrequent" anyway.')

,Id,Electrical,YearBuilt,Neighborhood
1379,1380,NaN,2006,Timber


Electrical among houses built 2000 or later: {'SBrkr': 1.0}
Modes learned from fold-1 training rows only: {'Electrical': 'SBrkr', 'MSZoning': 'RL', 'KitchenQual': 'TA'}
An "Unknown" level would be a column with 1 training row(s); OneHotEncoder(min_frequency=10) would merge it into "infrequent" anyway.


**Interpretation.** After structural absence is resolved, the training data has genuinely missing categories only in `MasVnrType` (8) and `Electrical` (1); the test file adds `MSZoning` (4), `Functional` (2), `KitchenQual` (1), `Exterior1st`/`Exterior2nd` (1 each) and more `MasVnrType` rows (16). Mode imputation is reasonable here because so few rows are affected, and the modes are dominant for `Electrical` (91%), `MSZoning` (79%) and `MasVnrType` (`None`, 60%, consistent with `MasVnrArea` imputed as 0). It is a weaker guess where the mode is not dominant (`KitchenQual` 50%, `Exterior1st` 35%); mode imputation can place a house in the wrong category, and the effect is small only because the counts are tiny. An explicit "Unknown" level was considered: for `Electrical` it would be a column with a single training row, which `min_frequency = 10` would pool into "infrequent" anyway. Id 1380 was built in 2006 and every house built since 2000 has `SBrkr`, which supports the mode. A formal comparison is not possible with one affected training row.

## 7. Neighborhood-Based Median Imputation
**Issue.** `LotFrontage` (street frontage in feet) is missing for 259 houses (17.7%). The EDA showed that the gaps are not random and that median frontage ranges from about 21 ft (BrDale, MeadowV) to about 91 ft (NoRidge).
**Method.** Two approaches are implemented and compared:
- **A. Global median**: one value for every missing house.
- **B. Neighborhood median**: the median of the training houses in the same neighbourhood; a neighbourhood with no observed frontage, or one never seen in training, falls back to the global training median.

Neighbourhood medians can preserve location-specific lot layouts (planned developments with narrow frontages, rural-style lots with wide ones). Whether this *improves* estimates is tested below rather than assumed: known frontages in each validation fold are hidden, imputed from the training fold, and compared with the true values.

In [12]:
print(inspect.getsource(NeighborhoodMedianImputer))
nm = NeighborhoodMedianImputer().fit(X.iloc[tr0])
med_tab = pd.DataFrame({'median LotFrontage (ft)': nm.medians_, 'n observed': X.iloc[tr0].groupby('Neighborhood').LotFrontage.count()}).round(1).sort_values('median LotFrontage (ft)')
print('Global training median (fold 1):', nm.global_median_)
display(med_tab.T)

class NeighborhoodMedianImputer(BaseEstimator, TransformerMixin):
    """Fill `column` with the median of its `group` (default Neighborhood), learned from training rows.
    A group with no observed value, or one never seen in training, falls back to the global training median."""

    def __init__(self, column="LotFrontage", group="Neighborhood"):
        self.column = column
        self.group = group

    def fit(self, X, y=None):
        self.global_median_ = float(X[self.column].median())
        self.medians_ = X.groupby(self.group)[self.column].median().dropna()
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        return self

    def transform(self, X):
        X = X.copy()
        fill = X[self.group].map(self.medians_).fillna(self.global_median_)
        X[self.column] = X[self.column].fillna(fill)
        return X

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_in_

Global training median (fold 1): 70.0


Neighborhood,BrDale,MeadowV,Blueste,NPkVill,Blmngtn,BrkSide,SWISU,IDOTRR,OldTown,StoneBr,Gilbert,SawyerW,Edwards,Veenker,CollgCr,Mitchel,Somerst,NAmes,Sawyer,Crawfor,NWAmes,ClearCr,Timber,NoRidge,NridgHt
median LotFrontage (ft),21.0,21.0,24.0,24.0,43.0,52.0,60.0,60.0,60.0,63.0,65.0,66.0,67.0,68.0,70.0,71.0,72.5,73.0,74.0,76.5,80.0,80.0,85.0,90.5,93.0
n observed,14.0,14.0,1.0,4.0,11.0,38.0,18.0,29.0,85.0,19.0,43.0,41.0,76.0,5.0,102.0,31.0,64.0,142.0,38.0,34.0,41.0,11.0,20.0,26.0,55.0


In [13]:
def recover(make, Xf, folds, col='LotFrontage'):
    # Hide known `col` values in each validation fold, impute them from the training fold, compare with the truth.
    rows = []
    for tr, va in folds:
        Xva = Xf.iloc[va]; keep = Xva[col].notna()
        hidden = Xva[keep].copy(); truth = hidden[col].values; hidden[col] = np.nan
        imp = make().fit(Xf.iloc[tr])
        pred = imp.transform(hidden)[col].values
        rows.append((np.sqrt(np.mean((pred - truth) ** 2)), np.mean(np.abs(pred - truth))))
    r = np.array(rows)
    return pd.Series({'RMSE (ft)': r[:, 0].mean(), 'RMSE sd': r[:, 0].std(ddof=1), 'MAE (ft)': r[:, 1].mean()})

rec = {'A. global median': recover(GlobalMedianImputer, X, folds),
       'B. neighborhood median': recover(NeighborhoodMedianImputer, X, folds)}
rec_ab = pd.DataFrame(rec).T.round(2)
rec_ab

,RMSE (ft),RMSE sd,MAE (ft)
A. global median,24.11,3.46,16.80
B. neighborhood median,22.09,3.17,13.33


**Interpretation.** In fold 1 the neighbourhood medians range from 21 ft (BrDale, MeadowV) to 93 ft (NridgHt), confirming the location pattern seen in the EDA; some groups have very few observed values (Blueste: 1, NPkVill: 4), so their medians are noisy. In the hidden-value test, neighbourhood medians recovered known frontages with RMSE 22.1 ft against 24.1 ft for the global median, and MAE 13.3 ft against 16.8 ft (fold-to-fold standard deviation of RMSE about 3 ft). This supports the neighbourhood approach as a more accurate imputer, in line with the EDA (F08). It measures imputation accuracy only; whether it helps the price model is tested in section 14.

## 8. KNN Imputation
**Issue.** Frontage is plausibly related to the size and character of the property, not only to its neighbourhood.
**Method.** `KNNFrontageImputer` estimates a missing frontage as the average frontage of the *k* most similar training properties. Similarity is measured on `LotArea`, `GrLivArea` and `OverallQual` (the EDA's main size, lot and quality signals); `SalePrice` is never used. Because KNN is distance-based, the features are first put on a common scale: areas are log-transformed (skew above 1.3, `LotArea` above 12) and every column is standardised with training statistics, otherwise square feet would swamp the 1-10 quality score. The imputed value is converted back to feet. KNN is applied to `LotFrontage` only. Its advantage is that it adapts to local similarity without a model; its limitations are sensitivity to the choice of features, *k* and scaling, and a cost that grows with the data. It is compared, not assumed better.

In [14]:
print(inspect.getsource(KNNFrontageImputer))

class KNNFrontageImputer(BaseEstimator, TransformerMixin):
    """Impute `target` from the k most similar training properties (similarity on `features`).

    Area columns are log1p-transformed (heavy right skew), all columns are standardised with
    training statistics so no feature dominates the distance, and the imputed value is converted
    back to the original unit. SalePrice is never used."""

    def __init__(self, target="LotFrontage", features=("LotArea", "GrLivArea", "OverallQual"),
                 log_cols=("LotArea", "GrLivArea"), n_neighbors=5, scale=True, weights="uniform"):
        self.target = target
        self.features = features
        self.log_cols = log_cols
        self.n_neighbors = n_neighbors
        self.scale = scale
        self.weights = weights

    def _matrix(self, X):
        cols = [self.target, *self.features]
        M = X[cols].astype(float).copy()
        for c in self.log_cols:
            M[c] = np.log1p(M[c])
        return M.values

    d

In [15]:
core = ('LotArea', 'GrLivArea', 'OverallQual')
extended = core + ('YearBuilt', '1stFlrSF', 'GarageArea')
cfgs = {}
for k in (3, 5, 7):
    cfgs[f'KNN k={k}, scaled, core features'] = lambda k=k: KNNFrontageImputer(n_neighbors=k, features=core)
cfgs['KNN k=5, NOT scaled, core features'] = lambda: KNNFrontageImputer(n_neighbors=5, features=core, scale=False)
cfgs['KNN k=5, scaled, extended features'] = lambda: KNNFrontageImputer(n_neighbors=5, features=extended)

class RatioImputer:               # reference from the EDA: frontage ~ ratio x sqrt(LotArea), ratio by neighbourhood
    def fit(self, Xt):
        r = Xt.LotFrontage / np.sqrt(Xt.LotArea); self.nb = r.groupby(Xt.Neighborhood).median(); self.g = r.median(); return self
    def transform(self, Xt):
        Xt = Xt.copy(); Xt['LotFrontage'] = Xt.LotFrontage.fillna(Xt.Neighborhood.map(self.nb).fillna(self.g) * np.sqrt(Xt.LotArea)); return Xt

rec_all = dict(rec)
for name, make in cfgs.items():
    rec_all[name] = recover(make, X, folds)
rec_all['reference: neighborhood ratio x sqrt(LotArea) (EDA)'] = recover(RatioImputer, X, folds)
rec_table = pd.DataFrame(rec_all).T.round(2).sort_values('RMSE (ft)')
rec_table.to_csv(TAB / 'prep_lotfrontage_recovery.csv'); rec_table

,RMSE (ft),RMSE sd,MAE (ft)
"KNN k=5, scaled, extended features",19.00,2.94,11.49
"KNN k=7, scaled, core features",19.01,3.27,11.79
"KNN k=5, NOT scaled, core features",19.15,3.13,11.82
reference: neighborhood ratio x sqrt(LotArea) (EDA),19.45,2.51,11.09
"KNN k=5, scaled, core features",19.57,3.31,12.04
"KNN k=3, scaled, core features",20.69,2.99,12.58
B. neighborhood median,22.09,3.17,13.33
A. global median,24.11,3.46,16.80


**Interpretation.** KNN recovered hidden frontages better than both median methods: RMSE 19.0 to 19.6 ft for the scaled five- and seven-neighbour variants (best 19.0), against 22.1 ft for neighbourhood medians and 24.1 ft for the global median; the EDA's ratio method (19.5 ft) is similar. Differences among the better KNN variants (about 0.5 ft) are small compared with the fold-to-fold spread (about 3 ft), so they do not justify preferring one; `k = 3` is clearly noisier (20.7 ft). Unscaled neighbours (19.2 ft) did as well as scaled ones in this case because areas are already log-transformed and the quality score has a similar spread; scaling is retained as a safeguard, not because it was shown to help here. `k = 5` is kept as the default because it is the middle value, not because it was best. These are imputation-accuracy results; their effect on price prediction is tested in section 14.

## 9. Missing-Indicator Features
**Issue.** After imputation a filled-in frontage looks identical to a measured one. The EDA showed that frontage is missing far more often on cul-de-sacs (52%) and irregular lots, so the fact that it was missing may itself carry information.
**Method.** `AddMissingIndicator` creates `LotFrontage_Missing` (1 = originally missing, 0 = observed) **before** imputation, and both columns are kept, so a model can tell a real measurement from an estimated one. Whether it helps is evaluated, not assumed: Strategy A (imputation only) is compared with Strategy B (imputation plus indicator) under identical folds.

In [16]:
ind = AddMissingIndicator(('LotFrontage',)).fit(X).transform(X)
imputed = NeighborhoodMedianImputer().fit(ind).transform(ind)
ex = pd.concat([X.LotFrontage.rename('Original LotFrontage'), imputed[['LotFrontage', 'LotFrontage_Missing']].rename(columns={'LotFrontage': 'After imputation'})], axis=1)
display(pd.concat([ex[ex['LotFrontage_Missing'] == 0].head(2), ex[ex['LotFrontage_Missing'] == 1].head(2)]))
print('Indicator sum:', int(ind.LotFrontage_Missing.sum()), '== original missing count:', int(X.LotFrontage.isna().sum()), '| values:', sorted(ind.LotFrontage_Missing.unique()))

,Original LotFrontage,After imputation,LotFrontage_Missing
0,65.0,65.0,0
1,80.0,80.0,0
7,NaN,80.0,1
12,NaN,71.0,1


Indicator sum: 259 == original missing count: 259 | values: [0, 1]


In [17]:
RESULTS = {}
def run(strategy, alpha=10, label=None, **kw):
    key = label or (strategy, alpha, tuple(sorted(kw.items())))
    if key not in RESULTS:
        model = Pipeline([('prep', build_preprocessor(strategy, **kw)), ('ridge', Ridge(alpha=alpha))])
        RESULTS[key] = cv_scores(model, X, y, folds)
    return RESULTS[key]

A, B = run('neighborhood'), run('neighborhood_indicator')
ind_cmp = pd.DataFrame({'A. imputation only': summarize(A), 'B. imputation + indicator': summarize(B)}).T
d, se = paired_difference(B, A)
ind_cmp['RMSE_mean'] = ind_cmp['RMSE_mean'].round(4); ind_cmp['RMSE_sd'] = ind_cmp['RMSE_sd'].round(4); ind_cmp[['MAE_mean', 'MAE_sd']] = ind_cmp[['MAE_mean', 'MAE_sd']].round(0)
display(ind_cmp)
print(f'Paired RMSE difference (B - A): {d:+.5f}  (standard error {se:.5f})')

,RMSE_mean,RMSE_sd,MAE_mean,MAE_sd
A. imputation only,0.131,0.0214,15919.0,1923.0
B. imputation + indicator,0.131,0.0214,15925.0,1909.0


Paired RMSE difference (B - A): -0.00001  (standard error 0.00003)


**Interpretation.** The indicator is created before imputation and its sum (259) equals the number of originally missing values; observed values are untouched and imputed values are flagged. In the comparison, imputation alone and imputation with the indicator both reach RMSE 0.1310 and MAE of about $15,900; the paired RMSE difference is -0.00001 with standard error 0.00003, i.e. no measurable benefit for this linear model. The indicator is therefore not selected by default, but it remains available as a strategy because a non-linear model could use the missingness pattern differently; this was not tested.

## 10. Categorical Encoding
**Issue.** Models need numbers, but encoding a category as an arbitrary integer invents an order.
**Method.** Variables are split by type:
- **Ordinal** (a real order exists): quality/condition ratings (`Po < Fa < TA < Gd < Ex`, with `NoFeature` lowest where absence is meaningful), basement exposure and finish, garage finish, lot shape, land slope, functionality, paved drive. `OrdinalEncoder` with an explicit category order.
- **Binary:** `CentralAir` (and `Street` if kept), encoded 0/1.
- **Nominal** (no order): `Neighborhood`, `HouseStyle`, `RoofStyle`, `Exterior1st`, etc. `OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=10)`: categories with fewer than 10 training rows are pooled into an "infrequent" column, which stabilises tiny groups (the EDA found neighbourhoods with 2 to 9 sales) and also absorbs levels never seen in training.
- **`MSSubClass`** is a numeric code for the dwelling class, so it is cast to text and one-hot encoded.
All encoders live inside the pipeline and are fitted on training folds only.

In [18]:
enc = pd.DataFrame({'type': ['Ordinal'] * len(P.ORDINAL_ORDERS) + ['Binary'] * len(P.BINARY_ORDERS) + ['Nominal'] * len(P.NOMINAL),
                    'feature': list(P.ORDINAL_ORDERS) + list(P.BINARY_ORDERS) + P.NOMINAL})
enc['order / levels (training)'] = [' < '.join(P.ORDINAL_ORDERS[f]) if f in P.ORDINAL_ORDERS else (' < '.join(P.BINARY_ORDERS[f]) if f in P.BINARY_ORDERS else f'{X[f].nunique()} levels') for f in enc.feature]
display(enc.groupby('type').agg(features=('feature', 'count')))
display(enc[enc.type != 'Nominal'].head(12))
code_vs_price = df.groupby('MSSubClass').SalePrice.median()
from scipy.stats import spearmanr
print('MSSubClass: numeric code vs median price, Spearman =', round(spearmanr(code_vs_price.index, code_vs_price.values)[0], 2), '-> no monotone order, so it is nominal')
display(code_vs_price.astype(int).to_frame('median price').T)

,features
type,
Binary,2
Nominal,21
Ordinal,19


,type,feature,order / levels (training)
0,Ordinal,ExterQual,Po < Fa < TA < Gd < Ex
1,Ordinal,ExterCond,Po < Fa < TA < Gd < Ex
2,Ordinal,HeatingQC,Po < Fa < TA < Gd < Ex
3,Ordinal,KitchenQual,Po < Fa < TA < Gd < Ex
4,Ordinal,BsmtQual,NoFeature < Po < Fa < TA < Gd < Ex
5,Ordinal,BsmtCond,NoFeature < Po < Fa < TA < Gd < Ex
6,Ordinal,FireplaceQu,NoFeature < Po < Fa < TA < Gd < Ex
7,Ordinal,GarageQual,NoFeature < Po < Fa < TA < Gd < Ex
8,Ordinal,GarageCond,NoFeature < Po < Fa < TA < Gd < Ex
9,Ordinal,PoolQC,NoFeature < Po < Fa < TA < Gd < Ex


MSSubClass: numeric code vs median price, Spearman = -0.06 -> no monotone order, so it is nominal


MSSubClass,20,30,40,45,50,60,70,75,80,85,90,120,160,180,190
median price,159250,99900,142500,107500,132000,215200,156000,163500,166500,140750,135980,192000,146000,88500,128250


In [19]:
prep0 = build_preprocessor('neighborhood_indicator').fit(X.iloc[tr0])
enc_ct = prep0.named_steps['encode']
sizes = {name: len(cols) for name, _, cols in enc_ct.transformers_ if name != 'remainder'}
out0 = prep0.transform(X.iloc[tr0])
print('Output columns by branch (input columns):', sizes, '| total encoded columns:', out0.shape[1])
print('KitchenQual mapping:', dict(zip(P.ORDINAL_ORDERS['KitchenQual'], range(len(P.ORDINAL_ORDERS['KitchenQual'])))))

Output columns by branch (input columns): {'numeric': 18, 'ordinal': 18, 'binary': 1, 'nominal': 19, 'flags': 2} | total encoded columns: 167
KitchenQual mapping: {'Po': 0, 'Fa': 1, 'TA': 2, 'Gd': 3, 'Ex': 4}


**Interpretation.** There are 19 ordinal, 2 binary and 21 nominal features (`MSSubClass` among the nominal ones). `MSSubClass` has no monotone relation with price (Spearman -0.06 between code and median price), so treating it as a number would be wrong. After the default pruning of low-information and redundant columns, the fold-1 pipeline (neighborhood_indicator strategy) encodes 18 numeric, 18 ordinal, 1 binary and 19 nominal input columns into 167 output columns. Rare categories (fewer than 10 training rows) share an "infrequent" column, and unseen categories at prediction time are routed there instead of failing. The EDA suggested testing ordinal against one-hot encoding for the rating scales; that comparison was not run and remains open.

## 11. Feature Engineering
**Issue.** Several raw columns describe one underlying property attribute (floor areas, bathroom counts, construction and remodel years), and some are exact or near-exact sums of each other.
**Method.** A small number of derived features, each built only from attributes known when a listing is prepared. `PropertyFeatureBuilder` implements them. Age is measured from a fixed valuation reference year, `2010` (the last year in the data), not from `YrSold`, because the eventual sale year is unknown at listing time and `SaleType`, `SaleCondition`, `MoSold` and `YrSold` are excluded as predictors.

In [20]:
fe = pd.DataFrame([
 ('TotalSF', 'TotalBsmtSF + 1stFlrSF + 2ndFlrSF', 'Total floor area including basement', 'One size measure; `GrLivArea` excludes the basement. Created, then removed by default as redundant (section 11b)', 'Physical attributes only'),
 ('TotalBathrooms', 'FullBath + 0.5*HalfBath + BsmtFullBath + 0.5*BsmtHalfBath', 'Bathroom capacity', 'Four sparse counts collapse to one interpretable count', 'Physical attributes only'),
 ('HouseAge', 'REFERENCE_YEAR (2010) - YearBuilt', 'Age at the valuation reference date', 'Age rather than a calendar year; replaces YearBuilt (exact linear transform)', 'Uses a fixed reference year, not the sale year'),
 ('RemodAge', 'REFERENCE_YEAR (2010) - YearRemodAdd', 'Years since last remodel or addition', 'Recency of renovation; replaces YearRemodAdd', 'As above'),
 ('IsRemodeled', 'YearRemodAdd != YearBuilt', 'Has the house been remodelled', 'Remodel flag the EDA did not capture in the raw years', 'Physical history only'),
], columns=['Feature', 'Formula', 'Business meaning', 'Reason for creation', 'Leakage considerations'])
display(fe)
src = inspect.getsource(PropertyFeatureBuilder)
print('Sale-context names used inside PropertyFeatureBuilder:', [c for c in P.SALE_CONTEXT if c in src] or 'none')

,Feature,Formula,Business meaning,Reason for creation,Leakage considerations
0,TotalSF,TotalBsmtSF + 1stFlrSF + 2ndFlrSF,Total floor area including basement,"One size measure; `GrLivArea` excludes the basement. Created, then...",Physical attributes only
1,TotalBathrooms,FullBath + 0.5*HalfBath + BsmtFullBath + 0.5*BsmtHalfBath,Bathroom capacity,Four sparse counts collapse to one interpretable count,Physical attributes only
2,HouseAge,REFERENCE_YEAR (2010) - YearBuilt,Age at the valuation reference date,Age rather than a calendar year; replaces YearBuilt (exact linear ...,"Uses a fixed reference year, not the sale year"
3,RemodAge,REFERENCE_YEAR (2010) - YearRemodAdd,Years since last remodel or addition,Recency of renovation; replaces YearRemodAdd,As above
4,IsRemodeled,YearRemodAdd != YearBuilt,Has the house been remodelled,Remodel flag the EDA did not capture in the raw years,Physical history only


Sale-context names used inside PropertyFeatureBuilder: none


In [21]:
Xs = StructuralAbsenceImputer().fit(X).transform(X)
Xf = PropertyFeatureBuilder().fit(Xs).transform(Xs)
lp = np.log1p(y)
chk = pd.DataFrame({'min': Xf[P.ENGINEERED].min(), 'median': Xf[P.ENGINEERED].median(), 'max': Xf[P.ENGINEERED].max(),
                    'NaN': Xf[P.ENGINEERED].isna().sum(),
                    'Spearman with log price': [spearmanr(Xf[c], lp)[0] for c in P.ENGINEERED]}).round(2)
display(chk)
print('For comparison - Spearman with log price: GrLivArea %.2f | FullBath %.2f | YearBuilt %.2f' % (spearmanr(X.GrLivArea, lp)[0], spearmanr(X.FullBath, lp)[0], spearmanr(X.YearBuilt, lp)[0]))

,min,median,max,NaN,Spearman with log price
TotalSF,334.0,2474.0,11752.0,0,0.82
TotalBathrooms,1.0,2.0,6.0,0,0.70
HouseAge,0.0,37.0,138.0,0,-0.65
RemodAge,0.0,16.0,60.0,0,-0.57
IsRemodeled,0.0,0.0,1.0,0,-0.09


For comparison - Spearman with log price: GrLivArea 0.73 | FullBath 0.64 | YearBuilt 0.65


**Interpretation.** The five derived features are complete (no missing values) and have sensible ranges. `TotalSF` correlates more strongly with log price (Spearman 0.82) than `GrLivArea` (0.73), and `TotalBathrooms` (0.70) more than `FullBath` (0.64), which supports combining them. `HouseAge` and `RemodAge` carry the same information as the raw years they replace (|rho| = 0.65 for `HouseAge`, as for `YearBuilt`); the reason for the change is the explicit reference year, not extra information. `IsRemodeled` is only weakly related to price (-0.09) and is a candidate for removal at feature selection. No derived feature uses a sale-context variable. `TotalSF` and, as parts, the four bathroom counts are later removed as redundant with the totals they form (section 11b); `TotalBathrooms` is kept. Because age is measured to a fixed year, it does not move with the sale year; this is an intentional simplification.

### 11b. Redundant and Correlated Features
**Issue.** The EDA found correlated pairs (`GarageCars`/`GarageArea` r = 0.88, `GrLivArea`/`TotRmsAbvGrd` 0.83, `TotalBsmtSF`/`1stFlrSF` 0.82) and exact sums (`GrLivArea` = `1stFlrSF` + `2ndFlrSF` + `LowQualFinSF`; `TotalBsmtSF` = its three basement parts). Redundant columns make linear coefficients unstable and split SHAP credit between duplicates arbitrarily.
**Method.** Keep the stronger-signal member of each correlated pair, and keep the totals instead of their parts. Columns removed (`REDUNDANT` in the module): `TotRmsAbvGrd`, `GarageArea`, `1stFlrSF`, `2ndFlrSF`, `BsmtFinSF1`, `BsmtFinSF2`, `BsmtUnfSF`, the four bathroom counts (parts of the engineered `TotalBathrooms`), and the engineered `TotalSF`, which equals `TotalBsmtSF` + `GrLivArea` once `LowQualFinSF` is dropped. A matrix-rank check confirmed that removing the first eight still left one exact dependency (`TotalBathrooms` and its four parts), so the bathroom parts were added. The effect is measured, not assumed.

In [22]:
print(pd.Series(P.REDUNDANT, name='reason for removal').to_frame().to_string())
def numeric_block(rr):
    p = build_preprocessor('baseline', reduce_redundancy=rr).fit(X)
    pre = Pipeline(p.steps[:-1]).transform(X)            # values before log transform and scaling
    cols = [c for c in pre.columns if c in P.NUMERIC]
    M = pre[cols].astype(float); M = M.fillna(M.median()).values      # diagnostic only: LotFrontage is still unimputed here
    Zs = (M - M.mean(0)) / M.std(0)
    vif = []
    for j in range(Zs.shape[1]):
        o = np.delete(Zs, j, axis=1); beta = np.linalg.lstsq(o, Zs[:, j], rcond=None)[0]
        r2 = 1 - np.sum((Zs[:, j] - o @ beta) ** 2) / np.sum(Zs[:, j] ** 2)
        vif.append(np.inf if r2 > 1 - 1e-9 else 1 / (1 - r2))
    return len(cols), np.linalg.matrix_rank(M), int(np.isinf(vif).sum()), max(v for v in vif if np.isfinite(v)), p.transform(X).shape[1]
red = pd.DataFrame({rr: numeric_block(rr) for rr in (False, True)}, index=['numeric columns', 'matrix rank', 'columns with infinite VIF', 'largest finite VIF', 'total output columns']).T
red.index = ['redundancy kept', 'redundancy removed (default)']
display(red.round(1))
keep_, drop_ = run('baseline', reduce_redundancy=False), run('baseline')
d_red, se_red = paired_difference(drop_, keep_)
print(f'Ridge RMSE: kept {keep_.rmse_log.mean():.5f}, removed {drop_.rmse_log.mean():.5f}; paired difference (removed - kept) {d_red:+.5f} (SE {se_red:.5f})')

                                                                                             reason for removal
TotRmsAbvGrd                                                                     r = 0.83 with GrLivArea (kept)
GarageArea                                                 r = 0.88 with GarageCars (kept; weaker price signal)
1stFlrSF                    part of GrLivArea (= 1stFlrSF + 2ndFlrSF + LowQualFinSF); r = 0.82 with TotalBsmtSF
2ndFlrSF                                                                                      part of GrLivArea
BsmtFinSF1                                          part of TotalBsmtSF (= BsmtFinSF1 + BsmtFinSF2 + BsmtUnfSF)
BsmtFinSF2                                                                                  part of TotalBsmtSF
BsmtUnfSF                                                                                   part of TotalBsmtSF
FullBath                   part of TotalBathrooms (= FullBath + 0.5 HalfBath + BsmtFullBath + 0.5 BsmtHa

,numeric columns,matrix rank,columns with infinite VIF,largest finite VIF,total output columns
redundancy kept,30.0,27.0,12.0,128.0,184.0
redundancy removed (default),18.0,18.0,0.0,3.8,172.0


Ridge RMSE: kept 0.13102, removed 0.13093; paired difference (removed - kept) -0.00010 (SE 0.00158)


**Interpretation.** Before removal, 30 numeric inputs had matrix rank 27, and 12 columns had infinite VIF: they were exact sums of one another (`GrLivArea` and its floors, `TotalBsmtSF` and its basement parts, `TotalBathrooms` and its four counts, `TotalSF`). After removing 12 redundant columns the numeric block (18 columns) has full rank and the largest VIF is 3.8. A rank check caught a dependency that the first version of this step missed: `TotalBathrooms` is an exact combination of the four bathroom counts, so those counts were removed too. Removing the redundant columns has no measurable effect on accuracy (paired Ridge RMSE difference -0.00010, standard error 0.00158), so the reason is interpretability: linear coefficients and additive attributions are no longer split arbitrarily between duplicate columns, which the proposal lists as a limitation of the decomposition. The weaker member of each pair was chosen by correlation with price. A further comparison on the same folds (outside this notebook) found that keeping `TotalBathrooms` and dropping its parts was slightly better than keeping the parts (Ridge -0.00025, standard error 0.00008; boosting -0.00045, standard error 0.00025).

## 12. Outlier Handling
**Issue.** Several size variables have extreme values. Removing a valid expensive or unusually large property would remove exactly the cases the model is meant to price, and would narrow the intervals artificially.
**Method.** Each extreme observation is judged on evidence (is it consistent with the property's other attributes and price?) and classified as valid, a data-entry mistake, or extreme-but-legitimate. Without reliable evidence of invalid data the observation is **retained**; heavy right tails are handled by `log1p` (section 13) rather than by deletion. The EDA findings are used, not repeated.

In [23]:
def fence_counts(s):
    q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
    return int((s > q3 + 1.5 * iqr).sum()), int((s > q3 + 3 * iqr).sum())
cols = ['LotArea', 'GrLivArea', 'LotFrontage', 'GarageArea', 'TotalBsmtSF']
rows = []
for c in cols:
    s = X[c].dropna(); a, b = fence_counts(s)
    top = df.loc[s.nlargest(3).index, 'Id'].tolist()
    rows.append((c, f'{s.median():,.0f}', f'{s.max():,.0f}', a, b, top))
display(pd.DataFrame(rows, columns=['feature', 'median', 'max', '> 1.5 IQR fence', '> 3 IQR fence', 'Ids of 3 largest']).set_index('feature'))
big = df[df.GrLivArea > 4000][['Id', 'GrLivArea', 'TotalBsmtSF', 'OverallQual', 'SaleCondition', 'SalePrice']].copy()
big['price per sq ft'] = (big.SalePrice / big.GrLivArea).round(0)
print('GrLivArea > 4000 (SaleCondition is shown for diagnosis only, it is not a predictor)'); display(big)
print('Three largest LotFrontage values (shape and configuration shown to judge plausibility):')
display(df.loc[df.LotFrontage.nlargest(3).index, ['Id', 'LotFrontage', 'LotArea', 'LotShape', 'LotConfig', 'Neighborhood', 'SalePrice']])

,median,max,> 1.5 IQR fence,> 3 IQR fence,Ids of 3 largest
feature,,,,,
LotArea,"9,478","215,245",67,34,"[314, 336, 250]"
GrLivArea,"1,464","5,642",31,4,"[1299, 524, 1183]"
LotFrontage,69,313,46,12,"[935, 1299, 1128]"
GarageArea,480,"1,418",21,3,"[1299, 582, 1191]"
TotalBsmtSF,992,"6,110",24,5,"[1299, 333, 497]"


GrLivArea > 4000 (SaleCondition is shown for diagnosis only, it is not a predictor)


,Id,GrLivArea,TotalBsmtSF,OverallQual,SaleCondition,SalePrice,price per sq ft
523,524,4676,3138,10,Partial,184750,40.0
691,692,4316,2444,10,Normal,755000,175.0
1182,1183,4476,2396,10,Abnorml,745000,166.0
1298,1299,5642,6110,10,Partial,160000,28.0


Three largest LotFrontage values (shape and configuration shown to judge plausibility):


,Id,LotFrontage,LotArea,LotShape,LotConfig,Neighborhood,SalePrice
934,935,313.0,27650,IR2,Inside,NAmes,242000
1298,1299,313.0,63887,IR3,Corner,Edwards,160000
1127,1128,182.0,14572,IR3,Corner,Gilbert,259000


In [24]:
OUTLIER_IDS = P.SUGGESTED_TRAINING_EXCLUSIONS
base = run('baseline')
with_ = cv_scores(Pipeline([('prep', build_preprocessor('baseline')), ('ridge', Ridge(alpha=10))]), X, y, folds, ids=df.Id.values, eval_exclude_ids=OUTLIER_IDS)
wo_ = cv_scores(Pipeline([('prep', build_preprocessor('baseline')), ('ridge', Ridge(alpha=10))]), X, y, folds, ids=df.Id.values, drop_train_ids=OUTLIER_IDS, eval_exclude_ids=OUTLIER_IDS)
outlier_sens = pd.DataFrame({'evaluate on all rows, train on all rows': summarize(base),
                             'evaluate without Ids 524/1299, train on all rows': summarize(with_),
                             'evaluate without Ids 524/1299, train without them': summarize(wo_)}).T.round(4)
outlier_sens[['MAE_mean', 'MAE_sd']] = outlier_sens[['MAE_mean', 'MAE_sd']].round(0)
display(outlier_sens)
d_out, se_out = paired_difference(wo_, with_)
print(f'Paired RMSE difference (drop - keep, same validation rows): {d_out:+.5f} (SE {se_out:.5f})')

,RMSE_mean,RMSE_sd,MAE_mean,MAE_sd
"evaluate on all rows, train on all rows",0.1309,0.0212,15910.0,1902.0
"evaluate without Ids 524/1299, train on all rows",0.1186,0.0113,15028.0,1191.0
"evaluate without Ids 524/1299, train without them",0.1144,0.0084,14391.0,1007.0


Paired RMSE difference (drop - keep, same validation rows): -0.00416 (SE 0.00100)


**Interpretation.** `LotArea` has the most extreme tail (67 lots beyond the 1.5 IQR fence, 34 beyond 3 IQR, maximum 215,245 sq ft) but these are plausible large lots and are handled by `log1p`. `LotFrontage` values of 313 ft (Ids 935 and 1299) occur on irregular (IR2, IR3) lots, so they are unusual but plausible, not evidence of entry errors; the area fields are also internally consistent (EDA F19). Ids 692 and 1183 are very large but priced at $175 and $166 per sq ft, consistent with their size, and are valid. Ids 524 and 1299 (4,676 and 5,642 sq ft) are `Partial` sales, i.e. unfinished homes, priced at $40 and $28 per sq ft; their records are not invalid but are not representative of what the attributes imply. Id 1299 is among the three largest values in four of the five examined features. Measured on a fixed Ridge, these two rows alone raise the cross-validated RMSE from 0.1186 to 0.1309 when included in validation, and training without them lowers the RMSE on the remaining rows from 0.1186 to 0.1144 (paired difference -0.00416, standard error 0.00100, about 4.2 standard errors). The pipeline keeps all rows; `SUGGESTED_TRAINING_EXCLUSIONS = [524, 1299]` in the module lets the modelling team exclude them from training, with results reported both ways. The final choice is group decision D2. No other observation is removed.

## 13. Feature Scaling
**Issue.** Features measured in square feet, counts and 1-10 scores have very different ranges. Scaling is required where the algorithm is sensitive to magnitude: **KNN** (distances, section 8) and **Ridge/Lasso/ElasticNet** (the penalty treats all coefficients alike, so unscaled large-range features are penalised unevenly). **Tree-based models** split on thresholds and are unaffected by monotone rescaling, so the pipeline can switch scaling off (`scale=None`).
**Method.** Scaling is applied to the numeric and ordinal columns only (one-hot and 0/1 flags are left as they are), inside the pipeline, fitted on training folds. Before scaling, the numeric columns whose *training* skew exceeds 0.75 are `log1p`-transformed by `SkewLog1p`; the column list is chosen in `fit`, so it cannot use validation data. `StandardScaler` (mean/standard deviation) and `RobustScaler` (median/IQR, less affected by extremes) are compared.

In [25]:
skew = prep0.named_steps['encode'].named_transformers_['numeric'].named_steps['skew']
names = prep0.named_steps['encode'].named_transformers_['numeric'].named_steps['skew'].feature_names_in_
print(f'{int(skew.mask_.sum())} of {len(names)} numeric columns are log1p-transformed (fold-1 training skew > 0.75):')
print(', '.join(names[skew.mask_]))

10 of 18 numeric columns are log1p-transformed (fold-1 training skew > 0.75):
LotFrontage, LotArea, MasVnrArea, TotalBsmtSF, GrLivArea, KitchenAbvGr, WoodDeckSF, OpenPorchSF, EnclosedPorch, ScreenPorch


In [26]:
scale_cmp = {}
for lbl, sc in [('StandardScaler', 'standard'), ('RobustScaler', 'robust'), ('no scaling', None)]:
    scale_cmp[lbl] = summarize(run('baseline', scale=sc))
scale_table = pd.DataFrame(scale_cmp).T
d_rs, se_rs = paired_difference(run('baseline', scale='robust'), run('baseline', scale='standard'))
d_sn, se_sn = paired_difference(run('baseline', scale='standard'), run('baseline', scale=None))
SCALE = 'robust' if -d_rs > 2 * se_rs else 'standard'          # same 2-standard-error rule as section 14
assert SCALE == inspect.signature(build_preprocessor).parameters['scale'].default, 'module default must match the evidence'
print(f'Paired RMSE difference robust - standard: {d_rs:+.5f} (SE {se_rs:.5f}); standard - none: {d_sn:+.5f} (SE {se_sn:.5f}) -> default scale = {SCALE!r}')
scale_table[['RMSE_mean', 'RMSE_sd']] = scale_table[['RMSE_mean', 'RMSE_sd']].round(4); scale_table[['MAE_mean', 'MAE_sd']] = scale_table[['MAE_mean', 'MAE_sd']].round(0)
display(scale_table)
for a_ in (3, 30):       # is the scaler ranking specific to alpha = 10?
    dd_, ss_ = paired_difference(run('baseline', alpha=a_, scale='robust'), run('baseline', alpha=a_, scale='standard'))
    print(f'alpha={a_}: paired RMSE difference robust - standard = {dd_:+.5f} (SE {ss_:.5f})')

Paired RMSE difference robust - standard: -0.00060 (SE 0.00026); standard - none: -0.00200 (SE 0.00041) -> default scale = 'robust'


,RMSE_mean,RMSE_sd,MAE_mean,MAE_sd
StandardScaler,0.1315,0.0221,15953.0,1931.0
RobustScaler,0.1309,0.0212,15910.0,1902.0
no scaling,0.1335,0.0223,16373.0,2022.0


alpha=3: paired RMSE difference robust - standard = -0.00028 (SE 0.00010)


alpha=30: paired RMSE difference robust - standard = -0.00092 (SE 0.00050)


**Interpretation.** In fold 1, 10 of the 18 numeric columns exceed the skew threshold and are log-transformed (lot frontage and area, veneer area, basement and living area, kitchen count, deck and porch areas). Scaling clearly helps Ridge: no scaling gives RMSE 0.1335, standard scaling 0.1315 and robust scaling 0.1309. Robust scaling is better than standard scaling by 0.00060 (standard error 0.00026, about 2.3 standard errors); the same direction holds at Ridge strengths 3 (2.8 standard errors) and 30 (1.8 standard errors, weaker). Applying the same two-standard-error rule used in section 14, `RobustScaler` is the default (`scale="robust"`); this is plausible because even after `log1p` several columns keep heavy tails that inflate a standard deviation. Sections 9 and 12 above already ran with this default, which this section justifies. Tree-based models do not need scaling and can use `scale=None`. The comparison uses one fixed Ridge strength and is not a model-selection result.

## 14. Preprocessing Strategy Comparison
**Issue.** Which way of handling `LotFrontage` gives the most useful inputs?
**Method.** Four strategies share the same structural-absence handling, other median/mode imputation, feature engineering and encoding, and differ only in `LotFrontage`:
1. **baseline:** global median.
2. **neighborhood:** neighbourhood median.
3. **neighborhood_indicator:** as 2, plus `LotFrontage_Missing`.
4. **knn:** KNN imputation (k = 5, scaled, core features).

A single **fixed** Ridge regression (alpha = 10, chosen in advance and not tuned) is used *only as a measuring instrument*; the actual model comparison belongs to the modelling stage. All strategies use the identical 15 folds (5-fold x 3 repeats, stratified by price band), with every preprocessing step fitted inside each training fold. Metrics: RMSE on log1p(SalePrice) and MAE in dollars.

**Decision rule, fixed before looking at results:** strategies are ordered by complexity (baseline < neighborhood < neighborhood_indicator < knn). A more complex strategy replaces a simpler one only if its mean paired RMSE improvement exceeds **two standard errors** of the paired difference; otherwise the simpler strategy is kept.

In [27]:
order = ['baseline', 'neighborhood', 'neighborhood_indicator', 'knn']
res = {s: run(s) for s in order}
strat = pd.DataFrame({s: summarize(res[s]) for s in order}).T
strat['paired dRMSE vs baseline'] = [paired_difference(res[s], res['baseline'])[0] for s in order]
strat['SE of dRMSE'] = [paired_difference(res[s], res['baseline'])[1] for s in order]
strat_table = strat.copy()
strat_table[['RMSE_mean', 'RMSE_sd', 'paired dRMSE vs baseline', 'SE of dRMSE']] = strat_table[['RMSE_mean', 'RMSE_sd', 'paired dRMSE vs baseline', 'SE of dRMSE']].round(5)
strat_table[['MAE_mean', 'MAE_sd']] = strat_table[['MAE_mean', 'MAE_sd']].round(0)
strat_table.to_csv(TAB / 'prep_strategy_comparison.csv'); display(strat_table)

# apply the pre-registered rule
selected = 'baseline'
for cand in order[1:]:
    d, se = paired_difference(res[cand], res[selected])
    if -d > 2 * se:
        selected = cand
print('Selected strategy:', selected)

,RMSE_mean,RMSE_sd,MAE_mean,MAE_sd,paired dRMSE vs baseline,SE of dRMSE
baseline,0.13093,0.02122,15910.0,1902.0,0.00000,0.00000
neighborhood,0.13101,0.02144,15919.0,1923.0,0.00009,0.00010
neighborhood_indicator,0.13101,0.02138,15925.0,1909.0,0.00008,0.00010
knn,0.13088,0.02115,15900.0,1893.0,-0.00004,0.00005


Selected strategy: baseline


In [28]:
# Robustness of the ranking: other Ridge strengths, and keeping the low-information columns
rob = {f'alpha={a}': {s: run(s, alpha=a).rmse_log.mean() for s in order} for a in (3, 10, 30)}
rob['alpha=10, keep low-information columns'] = {s: run(s, drop_low_information=False).rmse_log.mean() for s in order}
pd.DataFrame(rob).T.round(5)

,baseline,neighborhood,neighborhood_indicator,knn
alpha=3,0.13120,0.13128,0.13127,0.13111
alpha=10,0.13093,0.13101,0.13101,0.13088
alpha=30,0.13202,0.13214,0.13214,0.13204
"alpha=10, keep low-information columns",0.13224,0.13234,0.13234,0.13220


**Interpretation.** The four strategies are practically indistinguishable: mean RMSE on log price is 0.1309 to 0.1310 for all, MAE is about $15,900, and the paired differences against the baseline (+0.00009, +0.00008, -0.00004) are all within one standard error of zero (standard errors 0.00005 to 0.00010). No strategy beat the simpler one by two standard errors, so the pre-specified rule selects **baseline** (global-median imputation of `LotFrontage`). At Ridge strengths 3 and 30 the strategies also stay within about 0.0001 of each other. This sits alongside the imputation-accuracy results of sections 7 and 8, where KNN recovered frontage far better (19 to 20 ft against 24 ft): better imputation of one column did not improve price prediction. A plausible reason is that only 17.7% of rows are affected and that `LotArea`, neighbourhood and the other columns already carry most of the information frontage provides, but this was not tested. Keeping the eight low-information columns that the pipeline drops by default raises RMSE from 0.1309 to 0.1322, which supports removing them. KNN and neighbourhood strategies remain available in `build_preprocessor`, for example for a non-linear model.

## 15. Reusable Preprocessing Pipeline
**Issue.** The modelling team needs the same preprocessing for every model, without re-implementing it.
**Method.** `build_preprocessor(strategy, scale, drop_low_information, ...)` in `src/features/preprocessing.py` returns an **unfitted** scikit-learn `Pipeline` (DataFrame in, DataFrame out). Supported strategies: `baseline`, `neighborhood`, `neighborhood_indicator`, `knn`. The EDA's low-information columns are dropped (`Utilities`, `Street`, `Condition2`, `RoofMatl`: at least 98% one level; `PoolArea`, `3SsnPorch`, `LowQualFinSF`, `MiscVal`: at least 96% zero), as is `GarageYrBlt`; this can be switched off with `drop_low_information=False`.

In [29]:
pipe = build_preprocessor('neighborhood_indicator')
for name, step in pipe.steps:
    print(f'{name:12s}', type(step).__name__)
print()
print(inspect.signature(build_preprocessor))
print('Strategies:', P.STRATEGIES)
print('Dropped by default:', P.SALE_CONTEXT + P.IDENTIFIERS_AND_TARGET + P.ALWAYS_DROP_FEATURES + sorted(P.LOW_INFORMATION) + sorted(P.REDUNDANT))

guard        ColumnDropper
structural   StructuralAbsenceImputer
indicator    AddMissingIndicator
lotfrontage  NeighborhoodMedianImputer
features     PropertyFeatureBuilder
prune        ColumnDropper
encode       ColumnTransformer

(strategy='baseline', scale='robust', drop_low_information=True, knn_neighbors=5, min_frequency=10, skew_threshold=0.75, neighborhood_encoding='onehot', reduce_redundancy=True)
Strategies: ('baseline', 'neighborhood', 'neighborhood_indicator', 'knn')
Dropped by default: ['SaleType', 'SaleCondition', 'MoSold', 'YrSold', 'Id', 'SalePrice', 'GarageYrBlt', '3SsnPorch', 'Condition2', 'LowQualFinSF', 'MiscVal', 'PoolArea', 'RoofMatl', 'Street', 'Utilities', '1stFlrSF', '2ndFlrSF', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtFullBath', 'BsmtHalfBath', 'BsmtUnfSF', 'FullBath', 'GarageArea', 'HalfBath', 'TotRmsAbvGrd', 'TotalSF']


In [30]:
# How a model attaches to the pipeline (shown on fold 1 only; nothing is fitted on the full data here)
model = Pipeline([('prep', build_preprocessor('baseline')), ('ridge', Ridge(alpha=10))]).fit(X.iloc[tr0], np.log1p(y.iloc[tr0]))
pred = np.expm1(model.predict(X.iloc[va0]))
print('Fold-1 validation MAE ($):', round(float(np.mean(np.abs(pred - y.iloc[va0]))), 0))

Fold-1 validation MAE ($): 14750.0


**Interpretation.** The pipeline has a fixed sequence: guard (drop identifiers, target and sale-context columns) -> structural absence -> optional `LotFrontage` handling -> feature builder -> pruning of low-information columns -> encoding and scaling. Calling `build_preprocessor(strategy)` returns an unfitted object, so the modelling team can attach any estimator and fit it inside their own folds. The model example above is fitted on fold-1 training rows only.

## 16. Data Leakage Check
**Issue.** Leakage inflates accuracy silently, so each rule is tested rather than asserted.
**Method.** The pipeline is fitted on the training rows of fold 1 and compared with statistics from the full data; each check prints PASS or FAIL.

In [31]:
checks = []
def add(name, ok, detail): checks.append((name, 'PASS' if ok else 'FAIL', detail))

Xtr, Xva = X.iloc[tr0], X.iloc[va0]
p = build_preprocessor('neighborhood_indicator').fit(Xtr)
out_cols = list(p.transform(Xva).columns)
ct = p.named_steps['encode']

add('SalePrice and Id excluded from predictors', 'SalePrice' not in X and 'Id' not in X and not {'SalePrice', 'Id'} & set(out_cols), f'{len(out_cols)} output columns')
post = [c for c in out_cols if any(c.startswith(s) for s in P.SALE_CONTEXT)]
add('Post-sale variables excluded', not post, 'SaleType, SaleCondition, MoSold, YrSold absent from output')

pre = Pipeline(p.steps[:-1]).transform(Xtr)                       # frame entering the encoder, training rows only
num_imp = ct.named_transformers_['numeric'].named_steps['impute']; nn = list(ct.transformers_[0][2])
learned = pd.Series(num_imp.statistics_, index=nn)
full_med = Pipeline(p.steps[:-1]).transform(X)[nn].median()
gap = (learned - full_med).abs()
add('Median imputation statistics come from the training fold', np.allclose(learned.values, pre[nn].median().values) and (gap > 0).any(),
    f'learned medians equal training-row medians for all {len(nn)} numeric columns; {int((gap > 0).sum())} differ from full-data medians (max gap {gap.max():.1f})')

nm = p.named_steps['lotfrontage']
expected = Xtr.groupby('Neighborhood').LotFrontage.median().dropna()
full_nb = X.groupby('Neighborhood').LotFrontage.median().dropna()
diff = (full_nb - nm.medians_).abs().max()
add('Neighborhood medians learned from the training fold only', np.allclose(nm.medians_.reindex(expected.index), expected) and diff > 0, f'max difference from full-data medians: {diff:.1f} ft (non-zero, so validation rows were not used)')

pk = build_preprocessor('knn').fit(Xtr)
add('KNN imputer fitted on training fold only', pk.named_steps['lotfrontage'].knn_._fit_X.shape[0] == len(Xtr), f'donor rows = {pk.named_steps["lotfrontage"].knn_._fit_X.shape[0]} = training rows {len(Xtr)}')

ohe = ct.named_transformers_['nominal'].named_steps['onehot']
cats_fold = ohe.categories_[list(ohe.feature_names_in_).index('Neighborhood')]
add('Encoder categories learned from the training fold only', set(cats_fold) == set(Xtr.Neighborhood.unique()), f'{len(cats_fold)} neighbourhood categories in training fold')
rare = 'Blueste'; Xnr = Xtr[Xtr.Neighborhood != rare]
pn = build_preprocessor('neighborhood_indicator').fit(Xnr); outn = pn.transform(X[X.Neighborhood == rare])
add('Unseen category handled without error or leakage', outn.shape[0] > 0 and outn.isna().sum().sum() == 0 and pn.named_steps['lotfrontage'].medians_.get(rare) is None, f'{outn.shape[0]} {rare} rows transformed after removing the level from training')

sc = ct.named_transformers_['numeric'].named_steps['scale']
tr_out, va_out = p.transform(Xtr)[nn], p.transform(Xva)[nn]
tr_med, va_med = np.abs(tr_out.median()), np.abs(va_out.median())
add('Scaler fitted on the training fold only', bool((tr_med < 1e-9).all()) and bool((va_med > 1e-9).any()),
    f'RobustScaler centres training rows exactly (max |median| {tr_med.max():.1e}); validation rows are not centred exactly (max |median| {va_med.max():.2f})')

pi = build_preprocessor('neighborhood_indicator'); names_ = [n for n, _ in pi.steps]
ind_out = pi.fit(X).transform(X)['LotFrontage_Missing']
add('Missing indicator preserves original missingness', names_.index('indicator') < names_.index('lotfrontage') and int(ind_out.sum()) == int(X.LotFrontage.isna().sum()), f'indicator created before imputation; sum = {int(ind_out.sum())}')

src = inspect.getsource(PropertyFeatureBuilder)
add('Feature engineering uses no unavailable information', not [c for c in P.SALE_CONTEXT + ['SalePrice'] if c in src], f'fixed reference year {P.REFERENCE_YEAR}; inputs are physical attributes only')
leak = pd.DataFrame(checks, columns=['check', 'result', 'detail']); leak.to_csv(TAB / 'prep_leakage_checks.csv', index=False)
display(leak)
assert (leak.result == 'PASS').all()

,check,result,detail
0,SalePrice and Id excluded from predictors,PASS,167 output columns
1,Post-sale variables excluded,PASS,"SaleType, SaleCondition, MoSold, YrSold absent from output"
2,Median imputation statistics come from the training fold,PASS,learned medians equal training-row medians for all 18 numeric colu...
3,Neighborhood medians learned from the training fold only,PASS,"max difference from full-data medians: 4.5 ft (non-zero, so valida..."
4,KNN imputer fitted on training fold only,PASS,donor rows = 1168 = training rows 1168
5,Encoder categories learned from the training fold only,PASS,25 neighbourhood categories in training fold
6,Unseen category handled without error or leakage,PASS,2 Blueste rows transformed after removing the level from training
7,Scaler fitted on the training fold only,PASS,RobustScaler centres training rows exactly (max |median| 0.0e+00);...
8,Missing indicator preserves original missingness,PASS,indicator created before imputation; sum = 259
9,Feature engineering uses no unavailable information,PASS,fixed reference year 2010; inputs are physical attributes only


**Interpretation.** All ten checks pass. The evidence that statistics come from training rows only is specific: neighbourhood medians differ from the full-data medians by up to 4.5 ft; the median statistics of all 18 numeric columns equal the training-row medians while some differ from the full-data medians; the KNN donor set has exactly the 1,168 training rows; the robust scaler centres the training rows exactly but not the validation rows; and removing a neighbourhood from training (Blueste) leaves its validation rows transformable without error. The indicator is created before imputation, and no sale-context column or target appears in the output or in the feature builder. These checks cover the preprocessing; hyperparameter tuning and model fitting need their own review, which the proposal assigns to an independent group member.

## 17. Validate Preprocessing Output
**Issue.** The output must be complete, numeric, consistent and reusable.
**Method.** The selected strategy is fitted once on the full training data as a **demonstration artifact only** (it is not used for any evaluation; all reported performance in this notebook comes from fold-wise fits) and checked against 14 criteria. The unlabelled `test.csv` is used only to show that unseen records can be transformed.

In [32]:
sel = selected
demo = build_preprocessor(sel).fit(X)             # demonstration fit - NOT used for any reported score
Xin = X.copy(deep=True)
Z = demo.transform(X)
Zt = demo.transform(test_df.drop(columns=['SalePrice'], errors='ignore'))
v = []
def chk(n, ok, d=''): v.append((n, 'PASS' if ok else 'FAIL', d))

chk('1. Expected columns present, redundant columns absent', all(c in Z.columns for c in ['TotalBathrooms', 'HouseAge', 'RemodAge', 'IsRemodeled', 'LotArea', 'OverallQual', 'KitchenQual']) and not [c for c in P.REDUNDANT if c in Z.columns] and any(c.startswith('Neighborhood_') for c in Z.columns), f'{Z.shape[1]} columns')
chk('2. No unexpected missing values', int(Z.isna().sum().sum()) == 0 and int(Zt.isna().sum().sum()) == 0, f'train {int(Z.isna().sum().sum())}, test {int(Zt.isna().sum().sum())}')
chk('3. No infinite values', not np.isinf(Z.values).any(), '')
chk('4. Target excluded', 'SalePrice' not in Z.columns, '')
chk('5. Input DataFrame not modified', X.equals(Xin) and df.shape == (1460, 81), 'X identical after fit/transform')
loose = build_preprocessor('baseline', scale=None, skew_threshold=1e9).fit(X).transform(X)
chk('6. Original numeric values preserved where appropriate', np.allclose(loose['GrLivArea'], X.GrLivArea) and np.allclose(loose['OverallQual'], X.OverallQual) and np.allclose(loose['LotArea'], X.LotArea), 'unscaled, untransformed columns equal the input')
ordcols = [c for c in P.ORDINAL_ORDERS if c in loose.columns]
okord = all(loose[c].between(0, len(P.ORDINAL_ORDERS[c]) - 1).all() for c in ordcols)
nom_cols = [c for c in P.NOMINAL if c not in P.LOW_INFORMATION]
nom_ok = all(np.allclose(loose[[x for x in loose.columns if x.startswith(c + '_')]].sum(axis=1), 1) for c in nom_cols)
chk('7. Encoded categories valid', okord and nom_ok, 'ordinal codes within range; one-hot rows sum to 1')
nmv = build_preprocessor('neighborhood').fit(X); nmo = nmv.named_steps['lotfrontage']
probe = X[X.LotFrontage.isna()].head(3); filled = nmo.transform(probe).LotFrontage
chk('8. Neighborhood median imputation works', np.allclose(filled.values, probe.Neighborhood.map(nmo.medians_).fillna(nmo.global_median_).values) and filled.notna().all(), f'e.g. {probe.Neighborhood.iloc[0]} -> {filled.iloc[0]:.1f} ft')
kv = build_preprocessor('knn').fit(X); kf_ = kv.named_steps['lotfrontage'].transform(X)
chk('9. KNN imputation works', kf_.LotFrontage.notna().all() and np.allclose(kf_.LotFrontage[X.LotFrontage.notna()], X.LotFrontage.dropna()), f'imputed range {kf_.LotFrontage[X.LotFrontage.isna()].min():.0f}-{kf_.LotFrontage[X.LotFrontage.isna()].max():.0f} ft, observed values unchanged')
Zi = build_preprocessor('neighborhood_indicator').fit(X).transform(X)
chk('10. Missing indicator is binary', set(Zi.LotFrontage_Missing.unique()) <= {0, 1}, f'values {sorted(Zi.LotFrontage_Missing.unique())}')
chk('11. Unseen records can be transformed', Zt.shape == (len(test_df), Z.shape[1]) and list(Zt.columns) == list(Z.columns), f'{len(test_df)} test records -> {Zt.shape[1]} columns')
chk('12. Row count consistent', len(Z) == len(X) and len(Zt) == len(test_df), f'{len(X)} -> {len(Z)}')
chk('13. Feature names retrievable', len(demo.named_steps['encode'].get_feature_names_out()) == Z.shape[1], f'{Z.shape[1]} names')
Z2 = build_preprocessor(sel).fit(X).transform(X)
chk('14. Reproducible', Z.equals(Z2), 'two independent fits give identical output')
val = pd.DataFrame(v, columns=['check', 'result', 'detail']); val.to_csv(TAB / 'prep_validation_checks.csv', index=False)
display(val); assert (val.result == 'PASS').all()

,check,result,detail
0,"1. Expected columns present, redundant columns absent",PASS,172 columns
1,2. No unexpected missing values,PASS,"train 0, test 0"
2,3. No infinite values,PASS,
3,4. Target excluded,PASS,
4,5. Input DataFrame not modified,PASS,X identical after fit/transform
5,6. Original numeric values preserved where appropriate,PASS,"unscaled, untransformed columns equal the input"
6,7. Encoded categories valid,PASS,ordinal codes within range; one-hot rows sum to 1
7,8. Neighborhood median imputation works,PASS,e.g. NWAmes -> 80.0 ft
8,9. KNN imputation works,PASS,"imputed range 22-152 ft, observed values unchanged"
9,10. Missing indicator is binary,PASS,"values [0, 1]"


**Interpretation.** All 14 criteria pass on a demonstration fit of the selected strategy: complete, finite output of 172 columns, the target excluded, the input unchanged, valid encoded categories, working neighbourhood-median and KNN imputers, a binary indicator, consistent row counts, retrievable feature names, identical output from two independent fits, and successful transformation of all 1,459 unseen test records into the same 172 columns with no missing values. The demonstration fit is not used for any reported score.

## 18. Preprocessing Decision Log
Each row cites evidence computed in this notebook or recorded in the EDA insight log (`docs/eda_insight_log.md`, finding IDs in brackets). The log is also written to `docs/preprocessing_log.md`.

In [33]:
g_ = rec_table.loc['A. global median', 'RMSE (ft)']; n_ = rec_table.loc['B. neighborhood median', 'RMSE (ft)']
knn_best = rec_table[rec_table.index.str.startswith('KNN')]['RMSE (ft)']
ratio_ = rec_table.loc['reference: neighborhood ratio x sqrt(LotArea) (EDA)', 'RMSE (ft)']
sd_ = strat_table
dd = lambda s: strat_table.loc[s, 'paired dRMSE vs baseline']
dlog = [
 ('Structural missing values (14 columns)', "'NA' means the facility does not exist; most-frequent fill would assign facilities that are not there",
  'Most-frequent value; "Unknown" category; verified NoFeature', 'NoFeature when related columns confirm absence; facility-conditional mode when the facility exists',
  'Each NA is tied to a verified zero (garage, basement, fireplace, pool, misc) or to documentation (alley, fence)', 'Garage, basement, fireplace, pool and misc: every confirmed NA matches a zero; contradictions are Ids 333 and 949 in train and 14 group-level cases in test (2 garage, 8 basement, 3 pool, 1 misc) [F05]',
  'Alley and Fence cannot be cross-checked; absence taken from data_description.txt'),
 ('Numerical missing values', 'Zero-filling unknown measurements would invent absent facilities; mean is distorted by skew',
  'Zero; mean; median', 'Zero only for verified absence (Case A); median for unknown values (Case B)',
  'Median of MasVnrArea is 0 while its mean is 103.7 sq ft; LotArea skew 12.2', 'GarageArea has no NaN in train; Case B occurs in test (Id 2577) [F06, F09]',
  'Median ignores relationships with other columns'),
 ('Categorical mode imputation', 'A few genuinely missing categories (Electrical 1, MasVnrType 8; more in test)',
  'Mode; "Unknown" category', 'Most frequent category (SimpleImputer)',
  'Every house built since 2000 has SBrkr, and Id 1380 was built in 2006; an Unknown level would hold 1 training row and be pooled by min_frequency', 'Electrical SBrkr 91% [F06]',
  'The imputed value is an assumption, not the true category'),
 ('LotFrontage: neighborhood median', '17.7% missing, not missing at random', 'Global median; neighborhood median', 'Both implemented and tested; see comparison',
  f'Hidden-value recovery RMSE: global {g_:.1f} ft, neighborhood {n_:.1f} ft', 'Missingness tied to LotShape, Neighborhood, LotConfig [F07]', 'Neighborhoods with few observed values give noisy medians'),
 ('LotFrontage: KNN', 'Use property similarity rather than location only', 'KNN with k = 3, 5, 7; scaled / unscaled; core / extended features', 'Compared with the median methods',
  f'Best KNN recovery RMSE {knn_best.min():.1f} ft ({knn_best.idxmin()}); reference ratio method {ratio_:.1f} ft', 'See section 8', 'Depends on feature choice, k and scaling; not automatically better'),
 ('Missing-indicator feature', 'Imputed and measured frontages look identical afterwards', 'No indicator; LotFrontage_Missing', 'Compared; adopted only if it improves cross-validated error',
  f'Paired RMSE difference with vs without indicator: {dd("neighborhood_indicator") - dd("neighborhood"):+.5f}', 'Missing share 52% on cul-de-sacs vs 13% inside [F07]', 'Adds a column for little information if the difference is within noise'),
 ('Categorical encoding', 'Arbitrary integer codes would impose an order', 'Label encoding; one-hot; ordinal; target encoding', 'Ordinal for rating scales, one-hot (min_frequency = 10) for nominals, MSSubClass as nominal',
  'MSSubClass code has no monotone relation with price; rating scales have a natural order', 'Neighborhoods with 2-9 sales; unseen MSSubClass 150 in test [F12, F24]', 'Ordinal vs one-hot for the rating scales was not compared experimentally; target encoding of Neighborhood not tested (decision D3 open)'),
 ('Outliers', 'Extreme sizes could distort fits', 'Remove; cap; transform; keep', 'Keep all rows in the pipeline; log1p for skewed columns; optional exclusion of Ids 524 and 1299 left as a group decision',
  f'Ids 524 and 1299 are unfinished (Partial) sales priced far below their size; effect on Ridge CV RMSE of dropping them from training: {d_out:+.5f} (SE {se_out:.5f})', 'EDA F17, F18', 'Exclusion rests on out-of-model knowledge of the sale type (decision D2 open)'),
 ('Feature scaling', 'Mixed units; Ridge and KNN depend on magnitude', 'None; StandardScaler; RobustScaler', f'RobustScaler (paired RMSE gain over StandardScaler {-d_rs:.5f}, SE {se_rs:.5f}); scale=None for trees',
  f'RMSE: standard {scale_table.loc["StandardScaler", "RMSE_mean"]:.4f}, robust {scale_table.loc["RobustScaler", "RMSE_mean"]:.4f}, none {scale_table.loc["no scaling", "RMSE_mean"]:.4f}', 'Skewed areas [F10]', 'Compared with a single fixed Ridge strength'),
 ('Feature engineering', 'Related columns describe one attribute; years conflict with the unknown sale year', 'Raw columns; derived totals and ages', 'TotalSF, TotalBathrooms, HouseAge, RemodAge, IsRemodeled (reference year 2010)',
  'Derived features carry the same information in one interpretable column; no sale-context variable is used', 'Collinearity [F16]', 'Reference-year age is constant across sale years'),
 ('Correlated / redundant features', 'Correlated pairs and exact sums (GrLivArea, TotalBsmtSF and their parts) make coefficients and SHAP credit arbitrary', 'Keep all; keep one of each pair; keep totals not parts; also drop TotalSF',
  'Remove 12 columns (REDUNDANT): the weaker member of each pair, the parts of GrLivArea, TotalBsmtSF and TotalBathrooms, and TotalSF',
  f'Matrix rank deficiency removed (rank {int(red.loc["redundancy kept", "matrix rank"])} of {int(red.loc["redundancy kept", "numeric columns"])} -> {int(red.loc["redundancy removed (default)", "matrix rank"])} of {int(red.loc["redundancy removed (default)", "numeric columns"])}); Ridge RMSE change {d_red:+.5f} (SE {se_red:.5f})', 'EDA F16: r = 0.88, 0.83, 0.82; exact sums with infinite VIF', 'No accuracy gain was measured; the benefit is interpretability. The weaker member of a pair is chosen by correlation with price'),
 ('Leakage prevention', 'Fitted statistics must not see validation rows', 'Preprocess before splitting; preprocess inside folds', 'Everything is a Pipeline step, refitted in every fold',
  f'{int((leak.result == "PASS").sum())} of {len(leak)} leakage checks pass (section 16)', 'Sale-context cost +0.0008 RMSE on boosting probe [F22]', 'Checks cover the implemented steps; later model steps need their own review'),
]
log = pd.DataFrame(dlog, columns=['Feature / Issue', 'Problem Identified', 'Methods Considered', 'Selected Method', 'Justification', 'Evidence', 'Limitations'])
log.to_csv(TAB / 'prep_decision_log.csv', index=False)
md_lines = ['# Preprocessing Decision Log', '', 'Generated by `notebooks/02_preprocessing.ipynb` (section 18). Evidence IDs refer to `docs/eda_insight_log.md`.', '']
for _, r in log.iterrows():
    md_lines += [f'## {r["Feature / Issue"]}'] + [f'- **{k}:** {r[k]}' for k in log.columns[1:]] + ['']
(ROOT / 'docs' / 'preprocessing_log.md').write_text('\n'.join(md_lines))
display(log.style.set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap'}))

,Feature / Issue,Problem Identified,Methods Considered,Selected Method,Justification,Evidence,Limitations
0,Structural missing values (14 columns),'NA' means the facility does not exist; most-frequent fill would assign facilities that are not there,"Most-frequent value; ""Unknown"" category; verified NoFeature",NoFeature when related columns confirm absence; facility-conditional mode when the facility exists,"Each NA is tied to a verified zero (garage, basement, fireplace, pool, misc) or to documentation (alley, fence)","Garage, basement, fireplace, pool and misc: every confirmed NA matches a zero; contradictions are Ids 333 and 949 in train and 14 group-level cases in test (2 garage, 8 basement, 3 pool, 1 misc) [F05]",Alley and Fence cannot be cross-checked; absence taken from data_description.txt
1,Numerical missing values,Zero-filling unknown measurements would invent absent facilities; mean is distorted by skew,Zero; mean; median,Zero only for verified absence (Case A); median for unknown values (Case B),Median of MasVnrArea is 0 while its mean is 103.7 sq ft; LotArea skew 12.2,"GarageArea has no NaN in train; Case B occurs in test (Id 2577) [F06, F09]",Median ignores relationships with other columns
2,Categorical mode imputation,"A few genuinely missing categories (Electrical 1, MasVnrType 8; more in test)","Mode; ""Unknown"" category",Most frequent category (SimpleImputer),"Every house built since 2000 has SBrkr, and Id 1380 was built in 2006; an Unknown level would hold 1 training row and be pooled by min_frequency",Electrical SBrkr 91% [F06],"The imputed value is an assumption, not the true category"
3,LotFrontage: neighborhood median,"17.7% missing, not missing at random",Global median; neighborhood median,Both implemented and tested; see comparison,"Hidden-value recovery RMSE: global 24.1 ft, neighborhood 22.1 ft","Missingness tied to LotShape, Neighborhood, LotConfig [F07]",Neighborhoods with few observed values give noisy medians
4,LotFrontage: KNN,Use property similarity rather than location only,"KNN with k = 3, 5, 7; scaled / unscaled; core / extended features",Compared with the median methods,"Best KNN recovery RMSE 19.0 ft (KNN k=5, scaled, extended features); reference ratio method 19.4 ft",See section 8,"Depends on feature choice, k and scaling; not automatically better"
5,Missing-indicator feature,Imputed and measured frontages look identical afterwards,No indicator; LotFrontage_Missing,Compared; adopted only if it improves cross-validated error,Paired RMSE difference with vs without indicator: -0.00001,Missing share 52% on cul-de-sacs vs 13% inside [F07],Adds a column for little information if the difference is within noise
6,Categorical encoding,Arbitrary integer codes would impose an order,Label encoding; one-hot; ordinal; target encoding,"Ordinal for rating scales, one-hot (min_frequency = 10) for nominals, MSSubClass as nominal",MSSubClass code has no monotone relation with price; rating scales have a natural order,"Neighborhoods with 2-9 sales; unseen MSSubClass 150 in test [F12, F24]",Ordinal vs one-hot for the rating scales was not compared experimentally; target encoding of Neighborhood not tested (decision D3 open)
7,Outliers,Extreme sizes could distort fits,Remove; cap; transform; keep,Keep all rows in the pipeline; log1p for skewed columns; optional exclusion of Ids 524 and 1299 left as a group decision,Ids 524 and 1299 are unfinished (Partial) sales priced far below their size; effect on Ridge CV RMSE of dropping them from training: -0.00416 (SE 0.00100),"EDA F17, F18",Exclusion rests on out-of-model knowledge of the sale type (decision D2 open)
8,Feature scaling,Mixed units; Ridge and KNN depend on magnitude,None; StandardScaler; RobustScaler,"RobustScaler (paired RMSE gain over StandardScaler 0.00060, SE 0.00026); scale=None for trees","RMSE: standard 0.1315, robust 0.1309, none 0.1335",Skewed areas [F10],Compared with a single fixed Ridge strength
9,Feature engineering,Related columns descri

## 19. Final Preprocessing Summary

**1. Original dataset.** 1,460 rows and 81 columns (79 predictors after removing `Id` and `SalePrice`); the default pipeline outputs 172 model-ready columns from 57 source attributes; no duplicates; target without missing values and strongly right-skewed (1.88), modelled as `log1p(SalePrice)`.

**2. Main data quality issues.** `NA` means two different things (6,608 cells are documented absences, 357 are genuine gaps); 15 of 19 affected columns are structural; two contradictory basement records in training and 14 group-level contradictions in the test file; heavy right skew and mostly-zero areas; `MSSubClass` stored as a number; two unrepresentative unfinished-sale outliers.

**3. Missing-value decisions.** Structural absence is verified against related columns and becomes `NoFeature` (numeric twins 0); an `NA` on a facility that exists is filled with the most frequent category among existing facilities. Genuine numeric gaps use the training median, genuine categorical gaps the mode. `LotFrontage` was compared four ways; the selected strategy uses the global median.

**4. Categorical encoding.** Ordinal encoding with explicit order for 18 ordered variables (ratings, finishes, shape, slope, functionality), 0/1 for `CentralAir`, one-hot encoding with rare-level pooling (fewer than 10 rows) for 19 nominal variables including `MSSubClass`, with unseen categories handled.

**5. Outliers.** All rows retained by the pipeline; heavy tails handled by `log1p` and robust scaling. Ids 524 and 1299 are documented as unrepresentative (`SUGGESTED_TRAINING_EXCLUSIONS`); excluding them from training lowered Ridge RMSE by 0.0042 (about 4.2 standard errors) and boosting RMSE by 0.0015 (2.5 standard errors; separate experiment, `src/evaluation/decision_experiments.py`), whereas dropping two random rows gained nothing. The decision is left to the group (D2).

**6. Engineered features.** `TotalBathrooms`, `HouseAge`, `RemodAge` (from a fixed reference year of 2010) and `IsRemodeled` are kept; `TotalSF` is created but removed as redundant with `TotalBsmtSF` + `GrLivArea`. Twelve redundant or correlated columns are removed so that the numeric inputs have full rank (largest VIF 3.8); this has no measurable effect on accuracy. `SaleType`, `SaleCondition`, `MoSold` and `YrSold` are excluded.

**7. Scaling.** `RobustScaler` on numeric and ordinal columns (RMSE 0.1309 against 0.1315 for standard and 0.1335 for none), after `log1p` of columns whose training skew exceeds 0.75; `scale=None` for tree models.

**8. Strategy comparison.** Baseline, neighbourhood median, neighbourhood median with indicator and KNN gave RMSE 0.1309 to 0.1310 and MAE of about $15,900 under a fixed Ridge on 15 identical folds. Differences were within noise.

**9. Selected strategy.** `baseline`, by the rule fixed in advance (a more complex strategy must beat the simpler one by two standard errors); it is also the simplest. The other strategies remain available.

**10. Leakage prevention.** Every fitted quantity is learned inside each training fold; ten explicit checks pass (section 16).

**11. Remaining limitations.**
- The Ridge comparison cannot show whether non-linear models would benefit from the neighbourhood, indicator or KNN strategies.
- Ordinal against one-hot encoding of rating scales and target encoding of `Neighborhood` were not compared (decision D3 remains open).
- Mode imputation is an assumption (weak where the mode share is low).
- The outlier decision (D2) is open.
- Age uses a fixed reference year, not the real valuation date.
- Training data cover one city and 2006-2010.

**12. Reuse by the modelling team.** Run from the project root:
```python
from src.data.loading import load_raw
from src.features.preprocessing import build_preprocessor, SUGGESTED_TRAINING_EXCLUSIONS
from src.evaluation.cv import get_folds, cv_scores
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

df = load_raw()
X, y = df.drop(columns=["SalePrice", "Id"]), df["SalePrice"]
folds = get_folds(y)                                   # the shared 15 folds
model = Pipeline([("prep", build_preprocessor("baseline")),   # scale="robust" by default; scale=None for trees
                  ("model", Ridge(alpha=10))])
scores = cv_scores(model, X, y, folds, ids=df.Id.values)      # fits the preprocessor inside each training fold
```
The target is passed to `cv_scores` on the original scale and is converted to `log1p` inside it. Use `drop_train_ids=SUGGESTED_TRAINING_EXCLUSIONS` to exclude the two unrepresentative rows from training. Never call `fit` on the preprocessor with the full data before evaluation.